In [0]:
%pip install unidecode bs4 html2text openpyxl -qq

In [0]:
# Standard library
import gc
import json
import os
import re
import sys
import uuid
import functools
from datetime import datetime, date, timedelta
import html as html_escape
from decimal import Decimal
from pathlib import Path
from io import BytesIO

# Third-party - Data processing
from openpyxl import load_workbook
from dateutil.parser import parse
import pandas as pd

# Third-party - Text processing
import hashlib
import unicodedata
from bs4 import BeautifulSoup
from unidecode import unidecode

# Third-party - PySpark
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import DataFrame, SparkSession

# Third-party - Web & Security
import requests
import urllib3
from cryptography.fernet import Fernet

# Third-party - Type hints
from typing import Any, Callable, Dict, List, Literal, Optional, Protocol, Union

# Pandas compatibility patch
pd.DataFrame.iteritems = pd.DataFrame.items

In [0]:
key = "nxyxDEvwBQYIwGWpd7o4kgjvcdjpLWc4wGnl14adf84="
fernet = Fernet(key)

GFDOQA_TEST_PATH = "abfss://qa-automation@cacdgfdoqaautoadls.dfs.core.windows.net/tmp"

# abfss://{Container[layer]}@{StorageAccount[layer]}/{relative}
# GFDOQA_TEST_PATH = "abfss://gf-gaa-fmna-nonres-raw@cacgffrefitadlst.dfs.core.windows.net/QA_ATOM_Test"

TEST_NOT_INITIATED_MESSAGE = "Test not inited | Table property not set | Project name < 3 chars & desc not set < 10 chars."
COL_TEST_LOG = "No records in source and target, but col. test {result}."
input_filename = "input_file_name_by_spark"
#

#passwordmanager.py
class PasswordManager:
    @staticmethod
    def encrypt(plain_text:str):
        """Encrypt *plain_text* with the module Fernet key and return the token as str."""
        return fernet.encrypt(plain_text.encode()).decode("utf-8")

    @staticmethod
    def decrypt(encrypted_text:str):
        """Decrypt a Fernet token (str or bytes) created by encrypt(). Any other type returns "".

        Raises cryptography.fernet.InvalidToken when the token is not valid for the key.
        """
        decode = lambda txt: fernet.decrypt(txt).decode("utf-8")
        if isinstance(encrypted_text, bytes):
            return decode(encrypted_text)
        return decode(str.encode(encrypted_text)) if isinstance(encrypted_text,str) else ""

#settings.py
class SettingsProto(Protocol):
    enableUDF: bool
    replacelist:dict
    sprk: SparkSession
    loglevel:str
    writeformat:str
    applicationid:str
    summarylog_base:str
    errorlog_base:str
    maxerror:int


    def createTempViews(self, df, viewname) -> None: ...
    def cleartempview(self, viewname) -> None: ...


class Settings:
    CODE_VER = "1.0.0"

    def __init__(self):
        self.writeformat = "delta"
        self.projectname = ""
        self.projectdesc = ""
        self.maxerror = 10
        self.loglevel = "all"
        self.applicationid = ""
        self.output_root = ""
        self.errorlog_base = ""
        self.summarylog_base = ""
        self.replacelist = {}
        self.isdatabricks = False
        self.iswindows = False
        self.tempviews = []
        self.logfile = ""
        self.enableUDF = True
        self.vencrypted_password = True
        self.runallcols = False
        return

    def __str__(self):
        return f'Settings - Code version: {self.CODE_VER}.'

    def projectinfo(self,pname, pdesc):
        """Set the project name and description. inittest() requires name >= 3 and description >= 10 characters."""
        self.projectname = pname
        self.projectdesc = pdesc
        return

    def encrypted_password(self, password:str):
        """Despite its name, this sets a flag: when truthy (default True) loadrdbms() decrypts
        dbpassword with PasswordManager.decrypt(); when falsy the password is used as given.
        """
        self.vencrypted_password = password
        return

    def enableudf(self, value:bool):
        """Enable (True) or disable (False) the ATOM text cleaners; disabled cleaners return their input unchanged.

        The SQL functions read this value when they are registered (TestRunner.__init__), so call
        Atom_UDF.registeruserfunctions() again after changing it.
        """
        self.enableUDF = value
        return

    def setmaxerror(self, value):
        """Set the maximum number of error rows written by the Writer error logs.

        Values that cannot be converted to int fall back to 10.
        """
        try:
            self.maxerror = int(value)
        except:
            self.maxerror = 10
        return

    def setloglevel(self, value:Optional[str]):
        """Set the log level.

        "all": qalog, summary tables and error logs. "summary": qalog and summary tables (no error logs).
        "none": nothing is written. None or an unknown value falls back to "all".
        """
        default = "all"
        options = ["all", "summary", "none"]
        if value is not None and isinstance(value, str):
            value = value.lower()
            self.loglevel = value if value in options else default
        else:
            self.loglevel = default
        return

    def _set_directory(self, path:str):
        """Return *path* without one trailing slash or backslash."""
        cond = path.endswith("/") or path.endswith("\\")
        return path[:-1] if cond else path

    def seterrorlogfolder(self, path:str):
        """Set the error log folder. Note: TestRunner.__init__ overwrites it with {output_root}/{applicationid}/error_log."""
        self.errorlog_base = self._set_directory(path)
        return

    def setsummarylogfolder(self, path:str):
        """Set the summary log folder. Note: TestRunner.__init__ overwrites it with {output_root}/{applicationid}/summary_log."""
        self.summarylog_base = self._set_directory(path)
        return

    def getbasepath(self):
        return str(Path(__file__).parent)[:-23]

    def setwriteformat(self,f):
        """Set the format used to write logs (default "delta").

        For delta it also disables spark.databricks.delta.formatCheck, so setsparkobject() must be called first.
        """
        self.writeformat = str(f).strip()
        if self.writeformat.lower() == "delta":
            self.sprk.sql("SET spark.databricks.delta.formatCheck.enabled=false")
        return

    def setoutputpath(self, path:str):
        """Set the root output folder. Call it before creating TestRunner: the log folders are derived from it at init."""
        self.output_root = self._set_directory(path)
        return

    def setsparkobject(self, obj:SparkSession):
        """Set the SparkSession used by ATOM."""
        self.sprk = obj
        #self.db_utils = IPython.get_ipython().user_ns["dbutils"]
        return

    def createTempViews(self, df:DataFrame, viewname:str):
        """Create or replace the temp view *viewname* from *df*, mark *df* for caching (lazy) and
        track the name so cleartempviews() can drop it. Errors are logged, not raised.
        """
        try:
            df.createOrReplaceTempView(viewname)
            df.cache()
            self.tempviews.append(viewname)
        except Exception as e:
            Writer.logger(f"Exception in createTempViews: {e}")
        return

    def cleartempviews(self,excludelist=[]):
        """Drop every tracked temp view except those in *excludelist*."""
        for view in self.tempviews[:]:
            if view not in excludelist:
                self.cleartempview(view)
        return

    def cleartempview(self,viewname):
        """Drop the temp view *viewname* (if it exists) and stop tracking it. Errors are logged, not raised."""
        try:
            self.sprk.sql(f"drop view if exists {viewname}")
            if viewname in self.tempviews:
                self.tempviews.remove(viewname)
        except Exception as e:
            Writer.logger("exception in cleartempview")
            Writer.logger(e)
        return

    class Table:
        name = ''
        colTest = ''
        colTestDataType = ''
        dataTest = ''
        saveTable = ''
        debug = ''
        cleanPLLevel = '*'
        exColsInColTest = []
        exColsInDataTest = []
        srcPK = ''
        srcTableName = ''
        srcQuery = ''
        trgPK = ''
        trgTableName = ''
        trgQuery = ''
        failWhenMatch = ''

#api.py.....
class Rest:
    def __init__(self, settings:Union[Settings,None] = None):
        headers = {"Content-Type": "application/json; charset=UTF-8", "Accept": "application/json"}
        self.set_parameters("REST", "", "get", {}, {}, {}, {}, headers, 10000, 1, True)
        if settings is None:
            settings = Settings()
            settings.setsparkobject(SparkSession.builder.getOrCreate())
            print('Settings instance initiated.')
        self.settings = settings
        return

    def _set_data(self, d):
        return d if isinstance(d, dict) else ast.literal_eval(d)

    def set_parameters(
            self, source:str, endpoint:str, method:Literal['GET','POST'], data:dict, jsondata:dict,
            variables:dict, res_collection:dict, headers:dict, timeout:int, retry:int, ignore_ssl:bool
        ):
        self.__source = source
        self.__endpointURI = endpoint
        self.__apimethod = method
        self.__jdata = self._set_data(data)
        self.__jjson = self._set_data(jsondata)
        self.__variables = self._set_data(variables)
        self.__responsecollection = self._set_data(res_collection)
        self.__jheader = self._set_data(headers)
        self.__timeout = timeout
        self.__retryLimit = retry
        self.__ignoressl = ignore_ssl
        return

    def addheader(self, hn, hv):
        """Add or overwrite the request header *hn* with the value *hv*."""
        self.__jheader[hn] = hv

    def addauthtoken(self, token):
        """Set the header 'Authorization: Bearer <token>'."""
        self.__jheader["Authorization"] = f"Bearer {token}"
        return

    def addbasicauth(self, username, password):
        """Set the header 'Authorization: Basic base64(username:password)'.

        *password* must be a PasswordManager.encrypt() token; it is decrypted here.
        """
        token = base64.b64encode(f"{username}:{PasswordManager.decrypt(password)}".encode("utf-8")).decode("ascii")
        self.__jheader["Authorization"] = f"Basic {token}"
        return

    def __resolveparamaters(self):
        vars = self.__variables
        collection = self.__responsecollection
        set_vardata = lambda d, vd: ast.literal_eval(str(d).replace(f"${{{k}}}", vd))
        if len(vars)>0 and len(collection)>0:
            for k, v in vars.items():
                try:
                    vardata = JSONPath.read(collection[v["tablename"]], v["path"])
                except:
                    vardata = v["default"]
                # self.__jdata = re.sub(f"\\${{{k}}}", vardata, self.__jdata)
                self.__jdata = set_vardata(self.__jdata, vardata)
                self.__jheader = set_vardata(self.__jheader, vardata)
        return

    def getresponse(self):
        if self.__ignoressl:
            urllib3.disable_warnings()

        # resolve paramaters
        self.__resolveparamaters()
        if self.__apimethod.lower() == "get":
            method = lambda endpoint, d, json, h, v, t: requests.get(endpoint, h, v, t)
        else: 
            method = lambda endpoint, d, json, h, v, t: requests.post(endpoint, d, json, h, v, t)
        
        retry = 0
        success = False
        while retry<self.__retryLimit or not success:
            resp = method(
                self.__endpointURI, self.__jdata,self.__jjson,
                self.__jheader, (not self.__ignoressl), self.__timeout
            )
            success = resp.status_code in (200, 201)
            retry += 1
        return resp.json()

    def getDataFromPath(self, path):
        return JSONPath.read(self.__response, path)

    def getdataframe(self, responsedata = None):
        """Return the JSON response (or *responsedata*, if given) as a Spark DataFrame.

        Uses sparkContext, which is not available on standard (shared) or serverless compute.
        """
        spark_obj = self.settings.sprk
        res_data = json.dumps(self.getresponse() if responsedata is None else responsedata)
        return spark_obj.read.option("multiline", "true").json(spark_obj.sparkContext.parallelize([res_data]))

class JSONPath:
    # common vaiable for the class
    # jsonData = {}
    jsonDataList = [{}]
    jsonDataListCopy = [{}]
    logicalOperator = ""

    # read method================================================================================================
    @staticmethod
    def read(json, path):
        # JSONPath.jsonData = json
        o = m = None
        # regexp for split/ replace
        ARRAY_FILTER_CONDITION = "((?<=&&|\\|\\|)|(?=&&|\\|\\|))"
        ARRAY_FILTER = "\\=\\=(?=([^']*'[^']*')*[^']*$)|\\<\\=(?=([^']*'[^']*')*[^']*$)|\\<(?=([^']*'[^']*')*[^']*$)|\\>(?=([^']*'[^']*')*[^']*$)|\\>\\=(?=([^']*'[^']*')*[^']*$)|\\!\\=(?=([^']*'[^']*')*[^']*$)";
        PATH_ARRAY = "[.](?![^\\{\\[]*[\\]\\}])|[.](?![^\\{\\[]*[\\]\\}])"
        PATH_SEQ_ARRAY = "\\]\\[(?=(?:[^\\']*\\'[^\\']*\\')*[^\\']*$)"

        # variables for method
        path = re.sub(PATH_SEQ_ARRAY, "].[", path[2:])
        pathArray = re.compile(PATH_ARRAY).split(path)
        filtersPath = ""
        jsonListFilterConditions = None
        arrayFilterCtr = 0

        # loop on each path seperated by . ouside []
        # modify path if theer is array of array e.g. data[...][0] will be replaced to data[...].[0]
        for p in pathArray:
            filtersPath = ""
            # if path has [] that measn its is an array
            # seperate path and array filter
            if "[" in p and "]" in p:
                filtersPath = p[(p.index("[") + 1):]
                filtersPath = filtersPath[:-1]
                p = p[:p.index("[")]

            # if path is empty or main data is null or if it does not have the key leave temp o (object) untouched
            if len(p.strip()) > 0:
                if json is not None and json[p] is not None:
                    o = json[p]

            #
            if isinstance(o, dict):
                # if given path is json object then get the path and update the main map
                json = o
            elif isinstance(o, list):
                # if given path is jsonarray
                # get arrat from main json object
                JSONPath.jsonDataList = o

                # if array element is requested
                if len(filtersPath.strip()) > 0:
                    # if all elements of array is needed and an index or filter be used after
                    if json is None and len(p) > 0:
                        if isinstance(JSONPath.jsonDataList[0], dict):
                            p1 = p
                            m = [d[p1] for d in JSONPath.jsonDataList]
                            o = m
                            JSONPath.jsonDataList = o

                    # if array element is requested by index
                    if filtersPath.replace("\\]", "").isdigit():
                        o = JSONPath.jsonDataList[int(filtersPath)]
                        try:
                            json = o
                        except Exception as e:
                            pass
                    else:
                        # if array element is requested by conditions
                        # splt by each condition and=>&& or=>||
                        jsonListFilterConditions = list(
                            filter(None, re.compile(ARRAY_FILTER_CONDITION).split(filtersPath)))
                        arrayFilterCtr = 0
                        # process each condition
                        for condition in jsonListFilterConditions:
                            # remove spaces and the begining?
                            condition = condition.strip()
                            if condition.startswith("?"):
                                condition = condition[1:]

                            # remove unused (
                            while condition.startswith("(") and condition.endswith(")"):
                                condition = condition[1:-1]

                            # remove @.
                            if condition.startswith("@."):
                                condition = condition[2:]
                            # splt based on conditional operator, pass it to filter function
                            if "==" in condition:
                                JSONPath.__jsonListFilter(re.compile(ARRAY_FILTER).split(condition)[0],
                                                          re.compile(ARRAY_FILTER).split(condition)[-1],
                                                          "==", arrayFilterCtr)
                            elif "<=" in condition:
                                JSONPath.__jsonListFilter(re.compile(ARRAY_FILTER).split(condition)[0],
                                                          re.compile(ARRAY_FILTER).split(condition)[-1],
                                                          "<=", arrayFilterCtr)
                            elif "<" in condition:
                                JSONPath.__jsonListFilter(re.compile(ARRAY_FILTER).split(condition)[0],
                                                          re.compile(ARRAY_FILTER).split(condition)[-1],
                                                          "<", arrayFilterCtr)
                            elif ">=" in condition:
                                JSONPath.__jsonListFilter(re.compile(ARRAY_FILTER).split(condition)[0],
                                                          re.compile(ARRAY_FILTER).split(condition)[-1],
                                                          ">=", arrayFilterCtr)
                            elif ">" in condition:
                                JSONPath.__jsonListFilter(re.compile(ARRAY_FILTER).split(condition)[0],
                                                          re.compile(ARRAY_FILTER).split(condition)[-1],
                                                          ">", arrayFilterCtr)
                            elif "!=" in condition:
                                JSONPath.__jsonListFilter(re.compile(ARRAY_FILTER).split(condition)[0],
                                                          re.compile(ARRAY_FILTER).split(condition)[-1],
                                                          "!=", arrayFilterCtr)
                            elif condition == "&&" or condition == "||":
                                JSONPath.logicalOperator = condition

                            arrayFilterCtr = arrayFilterCtr + 1

                        # get the filtered array
                        o = JSONPath.jsonDataListCopy

                        # if list has only 1 element convert datatypes
                        # nullify other objects based on instance type

                        if len(JSONPath.jsonDataListCopy) == 1:
                            if isinstance(JSONPath.jsonDataListCopy[0], dict):
                                json = JSONPath.jsonDataListCopy[0]
                                o = json
                                JSONPath.jsonDataList = None
                            elif isinstance(JSONPath.jsonDataListCopy[0], list):
                                JSONPath.jsonDataList = JSONPath.jsonDataListCopy[0]
                                o = JSONPath.jsonDataList
                                json = None
                            elif isinstance(JSONPath.jsonDataListCopy[0], object):
                                o = JSONPath.jsonDataListCopy[0]
                                JSONPath.jsonDataList = None
                                json = None
                        else:
                            # if list has more then 1 element
                            # copy to master list object and nullify other objects
                            JSONPath.jsonDataList = JSONPath.jsonDataListCopy
                            json = None

                else:
                    # this is when all element from an array is the end
                    if json is None and len(p) > 0:
                        if isinstance(JSONPath.jsonDataList[0], dict):
                            p1 = p
                            m = [d[p1] for d in JSONPath.jsonDataList]
                            o = m
            else:
                # return if its is not map or list as it cannot go futher from here
                return str(o)

        # return the final result map or list or value
        return str(o)

    # read method================================================================================================

    @staticmethod
    def __jsonListFilter(colName, colValue, operator, ac):
        # emp mutable element
        jsonDataListNew = [{}]
        # to check if the value is numeric or non-numeric
        isString = "'" in colValue

        if isString:
            colValue = colValue[1:-1]

        # if this is the first condition
        # copy elements from master list to preocess
        # latter this master will be modifyed
        if ac == 0:
            JSONPath.jsonDataListCopy = JSONPath.jsonDataList.copy()
            JSONPath.logicalOperator = ""

        data = JSONPath.jsonDataList
        data_copy = JSONPath.jsonDataListCopy.copy()
        is_like = "*" in colValue or "%" in colValue
        colValueNorm = colValue if isString else float(colValue)
        numeric_ops = {
            "==": lambda a,b: a==b, "!=": lambda a,b: a!=b,
            "<=": lambda a,b: a<=b, ">=": lambda a,b: a>=b,
            "<": lambda a,b: a<b, ">": lambda a,b: a>b
        }
        # if the logical operator is && or the first one
        # from the copied master list filter based on conditions
        if JSONPath.logicalOperator == "" or JSONPath.logicalOperator == "&&":
            if isString:
                if operator == "==":
                    cond = (lambda e: colValueNorm in e[colName]) if is_like else (lambda e: colValueNorm == e[colName])
                    JSONPath.jsonDataListCopy = [d for d in data_copy if cond(d)]
                elif operator == "!=":
                    cond = (lambda e: colValueNorm not in e[colName]) if is_like else (lambda e: colValueNorm != e[colName])
                    JSONPath.jsonDataListCopy = [d for d in data_copy if cond(d)]
            else:
                if operator in numeric_ops:
                    op = numeric_ops[operator]
                    JSONPath.jsonDataListCopy = [d for d in data_copy if op(float(d[colName]), colValueNorm)]
        else:
            if isString:
                if operator == "==":
                    cond = (lambda e: colValueNorm in e[colName]) if is_like else (lambda e: colValueNorm == e[colName])
                    jsonDataListNew = [d for d in data if cond(d)]
                elif operator == "!=":
                    cond = (lambda e: colValueNorm not in e[colName]) if is_like else (lambda e: colValueNorm != e[colName])
                    jsonDataListNew = [d for d in data if cond(d)]
            else:
                if operator in numeric_ops:
                    op = numeric_ops[operator]
                    jsonDataListNew = [d for d in data if op(float(d[colName]), colValueNorm)]

        # merge results with ans and or
        data_copy = JSONPath.jsonDataListCopy.copy()
        JSONPath.jsonDataListCopy = list(filter(lambda e: e not in data_copy, jsonDataListNew)) + data_copy
        return
#api.py.....

#functions.py

# ---------------------------------------------------------------------------
# Pure text helpers.
# They live at module level on purpose: Spark pickles Python UDFs by value, and a
# function that references Atom_UDF would also serialize Atom_UDF._settings
# (including the SparkSession) into every UDF call.
# ---------------------------------------------------------------------------
_RE_WHITESPACE = re.compile(r"\s+")
_RE_PUNCTUATION = re.compile(r"[^\w\s]")
_RE_LEADING_ZEROS = re.compile(r"^0+(?!$)")

_DELETE_NEWLINES = str.maketrans("", "", "\r\n")
_DELETE_TAB = str.maketrans("", "", "\t")
_DELETE_TABLE_DELIMITERS = str.maketrans("", "", "\r\n\t")

_TS_LOG_FORMAT = "%Y-%m-%d %H:%M:%S.%f"
_TS_REPORT_FORMAT = "%b %d, %Y %H:%M:%S.%f"

_EXCEL_EPOCH = date(1899, 12, 30)
_EXCEL_MIN_SERIAL, _EXCEL_MAX_SERIAL = -693593, 2958465  # 0001-01-01 .. 9999-12-31

_JSONL_EXTENSIONS = (".jsonl", ".ndjson")
_COMPRESSION_EXTENSIONS = (".gz", ".bz2", ".zst", ".deflate", ".lz4", ".snappy")


def _is_jsonl_path(path: str) -> bool:
    """Return True when *path* points to a JSON Lines file (optionally compressed)."""
    name = str(path).lower().rstrip("/")
    for ext in _COMPRESSION_EXTENSIONS:
        if name.endswith(ext):
            name = name[: -len(ext)]
            break
    return name.endswith(_JSONL_EXTENSIONS)


def _safe_apply(fn: Callable[..., Any], s: Any, *args: Any) -> Any:
    """Apply fn(s, *args). On failure return s unchanged ("" when s is None)."""
    try:
        return fn(s, *args)
    except Exception:
        return "" if s is None else s


def _clean_newline(s: str) -> str:
    """Remove every CR and LF character."""
    return s.translate(_DELETE_NEWLINES)


def _clean_tab(s: str) -> str:
    """Remove every TAB character."""
    return s.translate(_DELETE_TAB)


def _clean_table_delimiters(s: str) -> str:
    """Remove every CR, LF and TAB character."""
    return s.translate(_DELETE_TABLE_DELIMITERS)


def _clean_double_space(s: str) -> str:
    """Collapse each whitespace run (spaces, tabs, line breaks) into one space. Leading/trailing spaces are kept."""
    return _RE_WHITESPACE.sub(" ", s)


def _clean_punctuations(s: str) -> str:
    """Replace each character that is not a letter, digit, underscore or whitespace with a space."""
    return _RE_PUNCTUATION.sub(" ", s)


def _trim_zero(s: str) -> str:
    """Remove leading zeros, keeping at least one character ("007" -> "7", "000" -> "0")."""
    return _RE_LEADING_ZEROS.sub("", s)


def _clean_spl_chrs(s: str) -> str:
    """Decompose accents (NFD) and drop every non-ASCII character ("\u00e9" -> "e", "\u20ac" -> "")."""
    return unicodedata.normalize("NFD", s).encode("ascii", "ignore").decode()


def _mask_hash(s: str) -> str:
    """Return the MD5 hex digest of *s* (compatible with Hive mask_hash)."""
    return hashlib.md5(s.encode()).hexdigest()


def _basic_clean(s: str) -> str:
    """Remove CR/LF/TAB, collapse whitespace and drop non-ASCII characters."""
    return _clean_spl_chrs(_clean_double_space(_clean_table_delimiters(s)))


def _clean_string(s: str) -> str:
    """_basic_clean() and then remove one pair of surrounding double quotes."""
    s = _basic_clean(s)
    return s[1:-1] if s.startswith('"') and s.endswith('"') else s


def _clean_all(s: str) -> str:
    """_basic_clean() and then replace punctuation with spaces (spaces are not collapsed again)."""
    # Punctuation (including quotes) is already replaced, so no quote stripping is needed.
    return _clean_punctuations(_basic_clean(s))


_RE_STRAY_BR = re.compile(r"</br\s*>", re.IGNORECASE)


def _html_to_text(s: str, tag_separator: str = "", decode_entities: bool = True) -> str:
    """Remove HTML tags; inner tags are replaced by *tag_separator* ("" = remove them).

    decode_entities=True  -> "&amp;" -> "&", "&lt;" -> "<", "&nbsp;" -> space (default).
    decode_entities=False -> tags are removed but entities are kept literally.
    """
    # Nothing to strip or decode.
    if "<" not in s and "&" not in s:
        return s
    if "<" in s:
        # html.parser drops a stray "</br>" and merges the adjacent text; treat it as "<br/>".
        s = _RE_STRAY_BR.sub("<br/>", s)
        # "&" is escaped so the parser never touches entities (it would turn "AT&T" into "ATT");
        # the parser turns "&amp;" back into "&", so the original entity text survives literally.
        # separator is inserted between text fragments (i.e. where an inner tag was).
        s = BeautifulSoup(s.replace("&", "&amp;"), features="html.parser").get_text(separator=tag_separator or "")
    # html.unescape decodes known entities and leaves unknown ones ("&T") untouched.
    return html_escape.unescape(s) if decode_entities else s


def _normalize(
    s: str, tag_separator: str = "", decode_entities: bool = True, transliterate: bool = True
) -> str:
    """Strip HTML (inner tags replaced by *tag_separator*), optionally transliterate to ASCII,
    drop control delimiters and collapse whitespace.

    decode_entities: decode HTML entities (&amp; &lt; &nbsp; ...). False keeps them literally.
    transliterate:   unidecode to ASCII ("e" for "\u00e9"). False preserves accents and symbols.
    """
    s = _html_to_text(s, tag_separator, decode_entities)
    if transliterate:
        s = unidecode(s)
    s = _clean_table_delimiters(s)
    # Also neutralize escaped sequences stored as literal text (backslash + r/n/t).
    s = s.replace("\\r", " ").replace("\\n", " ").replace("\\t", " ")
    return _RE_WHITESPACE.sub(" ", s)


def _replace_chr(s: str, find: str, replacement: str) -> str:
    """Replace every occurrence of *find* with *replacement*."""
    return s.replace(find, replacement)


def _replace_newline(s: str, replacement: str) -> str:
    """Replace each line break (CRLF, CR or LF) with *replacement*; CRLF counts as one."""
    return s.replace("\r\n", replacement).replace("\r", replacement).replace("\n", replacement)


def _replace_tab(s: str, replacement: str) -> str:
    """Replace every TAB with *replacement*."""
    return s.replace("\t", replacement)


def _replace_table_delimiters(s: str, replacement: str) -> str:
    """Replace line breaks (CRLF counts as one) and TABs with *replacement*."""
    return _replace_tab(_replace_newline(s, replacement), replacement)


def _replace_string(s: Optional[str], replacements: dict) -> str:
    """Apply each old -> new pair of *replacements* in order. None returns ""."""
    if s is None:
        return ""
    for old, new in replacements.items():
        try:
            s = s.replace(old, new)
        except Exception:
            pass
    return s


def _is_quoted(s: str) -> bool:
    """True when *s* starts and ends with a quote character (single or double; both ends may differ)."""
    return s[:1] in ("'", '"') and s[-1:] in ("'", '"')


# Ordered PK cleaning steps: level -> (condition or None, transformation).
# Level "3" (numeric formatting) never had any effect in the previous implementation
# ('{:.3f}'.format() was applied to a str and always failed), so it is kept as a no-op
# to preserve results and keep existing level strings such as "0134" valid.
_PK_STEPS: Dict[str, tuple] = {
    "0": (None, _clean_table_delimiters),
    "1": (None, _clean_spl_chrs),
    "2": (None, _clean_double_space),
    "4": (_is_quoted, lambda s: s[1:-1]),
}


def _pk(s: Any, level: Optional[str] = "*") -> Any:
    """Clean a primary key value. level selects steps (e.g. "012"); "*" or "all" runs every step."""
    if s is None:
        return ""
    level = "*" if level is None else str(level)
    run_all = "*" in level or "all" in level
    for step, (condition, transform) in _PK_STEPS.items():
        if run_all or step in level:
            try:
                if condition is None or condition(s):
                    s = transform(s)
            except Exception:
                pass
    return s


def _to_arraystring(arraystr: Any = "", delimiter: str = ";") -> Any:
    """Sort the items of an iterable (as str) and join them with *delimiter*. None returns "".

    A plain string is sorted character by character (legacy behavior).
    """
    # Legacy behavior: a plain string is sorted character by character.
    try:
        items = arraystr if isinstance(arraystr, str) else map(str, arraystr)
        return delimiter.join(sorted(items))
    except Exception:
        return "" if arraystr is None else arraystr


def _to_array(inputstr: Any = "", delimiter: str = ";") -> Any:
    """Split str(inputstr) by *delimiter* and return the sorted list. Note: None becomes ["None"]."""
    try:
        return sorted(str(inputstr).split(delimiter))
    except Exception:
        return [] if inputstr is None else inputstr


def _is_date(s: Any) -> bool:
    """True when dateutil can parse *s* as a date. Lenient: values such as "1" or "may" are accepted."""
    try:
        parse(s)
        return True
    except Exception:
        return False


def _atom_mask(s: Optional[str]) -> str:
    """Encrypt *s* with PasswordManager.encrypt(); None returns "".

    Not deterministic: the same value gives a different token on each call.
    """
    return "" if s is None else PasswordManager.encrypt(s)


def _get_timestamp_formatted(frmt: Optional[str] = _TS_LOG_FORMAT) -> str:
    """Format the current time with *frmt* and drop the last 3 characters.

    This turns %f microseconds into milliseconds; with formats not ending in %f it truncates the text.
    """
    return datetime.today().strftime(frmt or _TS_LOG_FORMAT)[:-3]


def _gated(fn: Callable[..., Any], enabled: bool) -> Callable[..., Any]:
    """Build a UDF body that applies fn safely when UDFs are enabled, or passes the input through."""
    if not enabled:
        return lambda s, *args: s
    return lambda s, *args: _safe_apply(fn, s, *args)


# ---------------------------------------------------------------------------
# DataFrame-level normalize support
# ---------------------------------------------------------------------------
_COMPLEX_TYPES = (T.StructType, T.ArrayType, T.MapType)
_STRING_TYPES = (T.StringType, T.VarcharType, T.CharType)
NormalizeMode = Literal["all", "selected", "applicable"]


@F.pandas_udf(T.StringType())
def _normalize_series(
    values: pd.Series, separators: pd.Series, decode_entities: pd.Series, transliterate: pd.Series
) -> pd.Series:
    """Vectorised pandas UDF that applies _normalize to every value in an Arrow batch."""
    return pd.Series(
        [
            _safe_apply(_normalize, v, sep, bool(dec), bool(tr))
            for v, sep, dec, tr in zip(values, separators, decode_entities, transliterate)
        ],
        index=values.index,
    )


class Atom_UDF:
    """ATOM helper functions: driver-side utilities and the UDFs used in SQL comparisons.

    Text cleaners honour Settings.enableUDF. When disabled, the input is returned unchanged.
    When enabled, a None input yields "" and a value that cannot be processed is returned as-is.
    """
    replacelist: dict = {}  # Kept for backward compatibility; replacestring() reads Settings.replacelist.
    _settings: Optional[SettingsProto] = None

    # Register the simple cleaners as temporary SQL functions (inlined by the optimizer and
    # executed natively by Photon) instead of row-by-row Python UDFs.
    use_native_sql: bool = True

    # ------------------------------------------------------------------ setup
    @classmethod
    def set_settings(cls, settings: SettingsProto) -> None:
        cls._settings = settings

    @classmethod
    def _enabled(cls) -> bool:
        return cls._settings is None or bool(cls._settings.enableUDF)

    @classmethod
    def _apply(cls, fn: Callable[..., Any], s: Any, *args: Any) -> Any:
        return _safe_apply(fn, s, *args) if cls._enabled() else s

    # ------------------------------------------------------- driver utilities
    @staticmethod
    def getNodeValue(node, key, defval):
        """Return the attribute *key* of an XML element, or *defval* if it is missing."""
        try:
            return node.attrib[key]
        except Exception:
            return defval

    @staticmethod
    def getNodeValueArr(node, index, key, defval):
        """Return the attribute *key* of the child node[index], or *defval* if it is missing."""
        try:
            return node[index].attrib[key]
        except Exception:
            return defval

    @staticmethod
    def getjsonValue(jobj, key, defval):
        """Return jobj[key], or *defval* if the key is missing."""
        try:
            return jobj[key]
        except Exception:
            return defval

    @staticmethod
    def getjsonbool(jobj, key, defval) -> bool:
        """Read jobj[key] as a bool: "true" (any case) or 1 -> True, any other value -> False. Missing key -> *defval*."""
        try:
            val = jobj[key]
        except Exception:
            return defval
        if isinstance(val, str):
            return val.lower() == "true"
        return isinstance(val, int) and val == 1

    @staticmethod
    def fieldname(t: str) -> str:
        """Turn a free-text header into a safe column name (ASCII, no punctuation, '_' separators)."""
        clean = lambda s: _clean_double_space(_clean_punctuations(_clean_spl_chrs(s))).replace(" ", "_")
        return _safe_apply(clean, t)

    @staticmethod
    def getCommonValues(a, b) -> list:
        """Case-insensitive intersection of two lists of strings, returned lowercased and in no particular order."""
        return list({s.lower() for s in a} & {s.lower() for s in b})

    @staticmethod
    def getdfschema(df: DataFrame) -> dict:
        """Map each lowercased column name to {"col_name", "col_datatype"} (simpleString). Returns {} on error."""
        try:
            return {
                f.name.lower(): {"col_name": f.name, "col_datatype": f.dataType.simpleString()}
                for f in df.schema.fields
            }
        except Exception:
            return {}

    @staticmethod
    def iif(e, t, f):
        """Inline if: return *t* when *e* is true, otherwise *f*.

        A str is true only when it equals "true" (any case); an int only when it equals 1.
        """
        if isinstance(e, str):
            e = e.lower() == "true"
        elif isinstance(e, int):
            e = e == 1
        return t if e else f

    @staticmethod
    def exceldate(n):
        """Convert an Excel serial date to 'YYYY-MM-DD'; out-of-range values are returned unchanged."""
        if _EXCEL_MIN_SERIAL <= n <= _EXCEL_MAX_SERIAL:
            return (_EXCEL_EPOCH + timedelta(days=n)).isoformat()
        return n

    @staticmethod
    def gettimestamp() -> dict:
        """Current time as {"log": "YYYY-MM-DD HH:MM:SS.mmm", "report": "Mon DD, YYYY HH:MM:SS.mmm"}."""
        now = datetime.today()
        return {
            "log": now.strftime(_TS_LOG_FORMAT)[:-3],
            "report": now.strftime(_TS_REPORT_FORMAT)[:-3],
        }

    @staticmethod
    def gettimestampf(frmt: Optional[str] = _TS_LOG_FORMAT) -> str:
        """Current time formatted with *frmt*, minus the last 3 characters (see _get_timestamp_formatted)."""
        return _get_timestamp_formatted(frmt)

    @staticmethod
    def getdate() -> str:
        """Current date as "Month DD, YYYY" (e.g. "October 28, 2026")."""
        return datetime.today().strftime("%B %d, %Y")

    # ----------------------------------------------------------- text cleaners
    @classmethod
    def htmlString(cls, inpString: Optional[str] = None, tag_separator: str = "",
                   decode_entities: bool = True, transliterate: bool = True):
        """Remove HTML tags/entities and normalize the resulting text.

        Args:
            tag_separator: Character(s) that replace inner HTML tags (default "" = remove them).
                           e.g. "<p>algo<br>como</br>esto</p>" with " " -> "algo como esto".
            decode_entities: Decode HTML entities (default True). False keeps "&amp;", "&lt;"... literally.
            transliterate: Transliterate to ASCII (default True). False preserves accents and symbols.
        """
        return cls._apply(_normalize, inpString, tag_separator, decode_entities, transliterate)

    @classmethod
    def normalize(cls, inpString: Optional[str] = None, tag_separator: str = "",
                  decode_entities: bool = True, transliterate: bool = True):
        """Strip HTML (inner tags replaced by *tag_separator*), optionally decode entities and
        transliterate to ASCII, remove \\r \\n \\t and collapse whitespace."""
        return cls._apply(_normalize, inpString, tag_separator, decode_entities, transliterate)

    @classmethod
    def cleanDoubleSpace(cls, inpString: Optional[str] = None):
        """Collapse each whitespace run into one space."""
        return cls._apply(_clean_double_space, inpString)

    @classmethod
    def mask_hash(cls, inpString: Optional[str] = None):
        """Hive-compatible mask_hash (MD5 hex digest)."""
        return cls._apply(_mask_hash, inpString)

    @classmethod
    def cleanNewLine(cls, inpString: Optional[str] = None):
        """Remove CR and LF characters."""
        return cls._apply(_clean_newline, inpString)

    @classmethod
    def cleanTab(cls, inpString: Optional[str] = None):
        """Remove TAB characters."""
        return cls._apply(_clean_tab, inpString)

    @classmethod
    def cleantabledelimiters(cls, inpString: Optional[str] = None):
        """Remove CR, LF and TAB characters."""
        return cls._apply(_clean_table_delimiters, inpString)

    @classmethod
    def cleanPunctuations(cls, inpString: Optional[str] = None):
        """Replace each character that is not a letter, digit, underscore or whitespace with a space."""
        return cls._apply(_clean_punctuations, inpString)

    @classmethod
    def trimzero(cls, inpString: Optional[str] = None):
        """Remove leading zeros, keeping at least one character."""
        return cls._apply(_trim_zero, inpString)

    @classmethod
    def cleanSplChrs(cls, inpString: Optional[str] = None):
        """Strip accents and drop every non-ASCII character."""
        return cls._apply(_clean_spl_chrs, inpString)

    @classmethod
    def cleanString(cls, inpString: Optional[str] = None):
        """Remove CR/LF/TAB, collapse whitespace, drop non-ASCII and strip one pair of surrounding double quotes."""
        return cls._apply(_clean_string, inpString)

    @classmethod
    def cleanAll(cls, inpString: Optional[str] = None):
        """Remove CR/LF/TAB, collapse whitespace, drop non-ASCII and replace punctuation with spaces."""
        return cls._apply(_clean_all, inpString)

    @classmethod
    def replacechr(cls, inpString: str, f: str, r: str):
        """Replace every occurrence of *f* with *r*."""
        return cls._apply(_replace_chr, inpString, f, r)

    @classmethod
    def replacenewline(cls, inpString: str, r: str):
        """Replace each line break (CRLF, CR or LF) with *r*."""
        return cls._apply(_replace_newline, inpString, r)

    @classmethod
    def replacetab(cls, inpString: str, r: str):
        """Replace every TAB with *r*."""
        return cls._apply(_replace_tab, inpString, r)

    @classmethod
    def replaceabledelimiters(cls, inpString: str, r: str):
        """Replace line breaks and TABs with *r*."""
        return cls._apply(_replace_table_delimiters, inpString, r)

    @classmethod
    def replacestring(cls, inpString: str):
        """Apply the Settings.replacelist old -> new pairs in order (none when UDFs are disabled). None returns ""."""
        replacements = cls._settings.replacelist if (cls._settings is not None and cls._enabled()) else {}
        return _replace_string(inpString, replacements or {})

    # ------------------------------------------------------------- other UDFs
    @staticmethod
    def atom_mask(s):
        """Encrypt *s* (see _atom_mask). Not deterministic, so masked values cannot be compared."""
        return _atom_mask(s)

    @staticmethod
    def isdate(s) -> bool:
        """True when *s* can be parsed as a date (lenient, see _is_date)."""
        return _is_date(s)

    @staticmethod
    def pk(inpString, pkLvl="*"):
        """Clean a key value. None returns "".

        *pkLvl* selects the steps: "0" remove CR/LF/TAB, "1" drop non-ASCII, "2" collapse whitespace,
        "3" no-op, "4" strip surrounding quotes. "*" or "all" (default) runs all of them.
        Steps always run in that order, whatever the order in *pkLvl*.
        """
        return _pk(inpString, pkLvl)

    @staticmethod
    def to_arraystring(arraystr="", delimiter=";"):
        """Sorted items joined by *delimiter* (see _to_arraystring)."""
        return _to_arraystring(arraystr, delimiter)

    @staticmethod
    def to_array(inputstr="", delimiter=";"):
        """Split by *delimiter* and sort (see _to_array)."""
        return _to_array(inputstr, delimiter)

    # ------------------------------------------------------- normalize columns
    @staticmethod
    def _quote(name: str) -> str:
        """Backtick-quote a column name so dots/spaces/special characters are handled safely."""
        return "`" + name.replace("`", "``") + "`"

    @classmethod
    def normalize_columns(
        cls,
        df: DataFrame,
        mode: NormalizeMode = "applicable",
        columns: Optional[List[str]] = None,
        exclude: Optional[List[str]] = None,
        tag_separator: str = "",
        decode_entities: bool = True,
        transliterate: bool = True,
    ) -> DataFrame:
        """Apply normalize() to the columns of a DataFrame.

        Args:
            df: Input DataFrame.
            mode:
                "all"        -> every simple-typed column (complex types are skipped so
                                 explode/flatten keep working; non-string simples are cast
                                 to string first).
                "selected"   -> only the columns listed in *columns*.
                "applicable" -> only string-typed columns; other columns are left untouched.
            columns: Column names to normalize (required when mode="selected"). Case-insensitive.
            exclude: Column names never to normalize (e.g. keys or audit columns). Case-insensitive.
            tag_separator: Character(s) that replace inner HTML tags (default "" = remove them).
            decode_entities: Decode HTML entities (default True). False keeps them literally.
            transliterate: Transliterate to ASCII (default True). False preserves accents and symbols.

        Returns:
            A DataFrame with the same columns and order, normalized where applicable.
        """
        mode = (mode or "").lower()
        if mode not in ("all", "selected", "applicable"):
            raise ValueError(f"Invalid mode '{mode}'. Use 'all', 'selected' or 'applicable'.")
        if not cls._enabled():
            return df

        fields = df.schema.fields  # Read schema once (avoids repeated Analyze RPCs).
        by_lower = {f.name.lower(): f.name for f in fields}
        excluded = {c.lower() for c in (exclude or [])}

        if mode == "selected":
            if not columns:
                raise ValueError("mode='selected' requires a non-empty 'columns' list.")
            missing = [c for c in columns if c.lower() not in by_lower]
            if missing:
                raise ValueError(f"Columns not found in DataFrame: {missing}")
            targets = {c.lower() for c in columns}
        elif mode == "all":
            # Complex columns are left untouched so explode/flatten keep working.
            targets = {f.name.lower() for f in fields if not isinstance(f.dataType, _COMPLEX_TYPES)}
        else:  # applicable
            targets = {f.name.lower() for f in fields if isinstance(f.dataType, _STRING_TYPES)}

        targets -= excluded

        # Single select keeps the plan flat and preserves column order.
        separator = F.lit(tag_separator or "")
        decode = F.lit(bool(decode_entities))
        translit = F.lit(bool(transliterate))
        projection = [
            _normalize_series(F.col(cls._quote(f.name)).cast("string"), separator, decode, translit).alias(f.name)
            if f.name.lower() in targets
            else F.col(cls._quote(f.name))
            for f in fields
        ]
        return df.select(*projection)

    # ------------------------------------------------------------ registration
    @staticmethod
    def _native_sql_functions(enabled: bool) -> Dict[str, str]:
        """Temporary SQL functions with the same semantics as the Python cleaners (NULL -> '')."""
        cr, lf, tab = "char(13)", "char(10)", "char(9)"
        crlf = f"concat({cr}, {lf})"
        one_arg, two_args = "s STRING", "s STRING, r STRING"
        definitions = {
            "cleantab": (one_arg, f"coalesce(translate(s, {tab}, ''), '')"),
            "cleannewline": (one_arg, f"coalesce(translate(s, {crlf}, ''), '')"),
            "cleantabledelimiters": (one_arg, f"coalesce(translate(s, concat({cr}, {lf}, {tab}), ''), '')"),
            "trimzero": (one_arg, "coalesce(regexp_replace(s, '^0+(?!$)', ''), '')"),
            "mask_hash": (one_arg, "coalesce(md5(s), '')"),
            # A NULL replacement leaves the input unchanged, like the Python version.
            "replacetab": (two_args, f"coalesce(replace(s, {tab}, r), s, '')"),
            "replacenewline": (
                two_args, f"coalesce(replace(replace(replace(s, {crlf}, r), {cr}, r), {lf}, r), s, '')"
            ),
            "replaceabledelimiters": (
                two_args,
                f"coalesce(replace(replace(replace(replace(s, {crlf}, r), {cr}, r), {lf}, r), {tab}, r), s, '')",
            ),
        }
        return {
            name: (
                f"CREATE OR REPLACE TEMPORARY FUNCTION {name}({params}) "
                f"RETURNS STRING RETURN {body if enabled else 's'}"
            )
            for name, (params, body) in definitions.items()
        }

    @classmethod
    def registeruserfunctions(cls) -> None:
        """Register all ATOM functions in the Spark session.

        Settings (enableUDF, replacelist) are captured at registration time, so call this
        again after changing them.
        """
        if cls._settings is None:
            raise ValueError("Atom_UDF.set_settings() must be called before registeruserfunctions().")
        print("registering udf functions....")
        spark = cls._settings.sprk
        enabled = bool(cls._settings.enableUDF)
        replacements = dict(cls._settings.replacelist or {})

        native = set()
        if cls.use_native_sql:
            for name, statement in cls._native_sql_functions(enabled).items():
                try:
                    spark.sql(statement)
                    native.add(name)
                except Exception as e:
                    print(f"Native SQL function '{name}' not created, falling back to Python UDF: {e}")

        gated_string_udfs = {
            "mask_hash": _mask_hash,
            "cleantabledelimiters": _clean_table_delimiters,
            "htmlstring": _normalize,
            "cleandoublespace": _clean_double_space,
            "cleanstring": _clean_string,
            "cleantab": _clean_tab,
            "cleannewline": _clean_newline,
            "cleanpunctuations": _clean_punctuations,
            "trimzero": _trim_zero,
            "cleansplchrs": _clean_spl_chrs,
            "normalize": _normalize,
            "cleanall": _clean_all,
            "replacechr": _replace_chr,
            "replacenewline": _replace_newline,
            "replacetab": _replace_tab,
            "replaceabledelimiters": _replace_table_delimiters,
        }
        active_replacements = replacements if enabled else {}
        python_udfs = {
            name: (_gated(fn, enabled), T.StringType())
            for name, fn in gated_string_udfs.items()
            if name not in native
        }
        python_udfs.update({
            "pk": (_pk, T.StringType()),
            "to_array": (_to_array, T.ArrayType(T.StringType())),
            "to_arraystring": (_to_arraystring, T.StringType()),
            "isdate": (_is_date, T.BooleanType()),
            "atom_mask": (_atom_mask, T.StringType()),
            "gettimestamp": (_get_timestamp_formatted, T.StringType()),
            "replacestring": (lambda s: _replace_string(s, active_replacements), T.StringType()),
        })
        for name, (fn, return_type) in python_udfs.items():
            spark.udf.register(name, fn, return_type)

        print(f"registering udf functions.... completed ({len(native)} native SQL, {len(python_udfs)} Python)")

#functions.py

#testrunner.py
class TestRunner:
    _backend_path:str = GFDOQA_TEST_PATH
    def __init__(self, settings:Optional[Settings] = None):
        # INIT_STATUS = True
        self.__src_trg_file_list=[]
        numbr = str(datetime.today().strftime("_%H.%M.%S.%f"))
        yyyymm = str(datetime.today().strftime("%Y%m"))
        # self.__applicationid = str(settings.sprk.sparkSessionId()) + numbr
        self.__applicationid = "app_" + yyyymm + "_" + str(uuid.uuid4()) + numbr
        
        if settings is None:
            settings = Settings()
            settings.setsparkobject(SparkSession.builder.getOrCreate())
            print('Settings instance initiated.')
        self._settings = self.__set_settings(settings)
        # This ensures Settings instance is injested into Atom_UDF class and its static and class methods
        Atom_UDF.set_settings(self._settings)
        Writer.set_settings(self._settings)
        

        banner_text = f"""
            _  _____ ___  __  __
           / \\|_   _/ _ \\|  \\/  |
          / _ \\ | || | | | |\\/| |
         / ___ \\| || |_| | |  | |
        /_/   \\_\\_| \\___/|_|  |_|    version {self._settings.CODE_VER}.
        spark version {self._settings.sprk.version}.
        python version {sys.version}.\n
        """

        Writer.logger(banner_text, True)

        self.__dqId = 0
        self.__dqcols = []

        #Set private variables
        self.__resdetailsummary = {}
        self.__dqcols = []
        self.__dqId = 0
        self.__ressummary = {}
        self.__responsecollection = {}
        self.__username = ""
        self.__applicationid = ""
        self.__curdate = None
        self.__src_trg_file_list = []
        self.__test_init=False
        self.__tableprop_status=False

        self.__tableprops = {
            "TABLENAME": "", "COLTEST": True, "COLTEST-DATATYPE": True, "DATATEST": True,
            "SAVETABLE": True, "DEBUG": False, "CLEANPKLEVEL": "*", "REPARTITION": False,
            "EXCLUDECOLS-COLTEST": [input_filename], "EXCLUDECOLS-DATATEST": [input_filename],
            "REVERSECHK": False, "FILE_NAME": "NA"
        }

        Atom_UDF.replacelist = self._settings.replacelist
        Atom_UDF.registeruserfunctions()
        Writer.logger(f"runid={self._settings.applicationid}", True)
        self.loginfo()
        return
    
    # Set initial configuration for settings
    def __set_settings(self, settings:Settings) -> Settings:
        settings.applicationid = self.__applicationid
        settings.errorlog_base = f"{settings.output_root}/{settings.applicationid}/error_log"
        settings.summarylog_base = f"{settings.output_root}/{settings.applicationid}/summary_log"

        settings.isdatabricks = ("DATABRICKS_RUNTIME_VERSION" in os.environ)
        settings.iswindows = ("Windows_NT" in os.environ)
        if settings.iswindows:
            self.__username = os.getlogin()
        else:
            self.__username = settings.sprk.sql("select current_user()").first()[0]
        settings.username = self.__username
        return settings

    # get samples from dataframe as string
    def __getDFSamples(self, df:DataFrame, samples:int = 1):
        """Return the first *samples* rows of *df* as a text table (collected to the driver)."""
        return df.limit(samples).toPandas().to_string(justify="left")

    # get schema from dataframe
    def __getDFSchema(self, df:DataFrame):
        """Return the schema of *df* as a simpleString."""
        return df.schema.simpleString()

    # add missing default options
    def __update_dict(self, useroptions:dict, defoptions:dict) -> dict:
        """Merge *useroptions* over *defoptions* (user values win). Mutates and returns *defoptions*."""
        defoptions.update(useroptions)
        return defoptions

    # log
    def loginfo(self):
        self.__curdate = Atom_UDF.gettimestamp()
        Writer.clear()
        Writer.add(self._settings.applicationid)
        Writer.add(self.__curdate["log"])
        Writer.add(self._settings.projectname)
        Writer.add(self._settings.projectdesc)
        Writer.add(self.__username)
        Writer.add(self._settings.CODE_VER)
        Writer.add(self.__curdate["log"][:4])
        Writer.qa_testinfo()

    def getsummary(self):
        """Return the run-level summary dict (test and error counts)."""
        return self.__ressummary

    def settableproperty(
            self, tablename:str = "", coltest:bool = True, coldatatypetest:bool = True, datatest:bool = True, savetable:bool = True,
            debug:bool = False, cleanpklevel:str = "*", repartition:bool = False, excolsincoltest:list = [],
            excolsindatatest:list = [], reversecheck:bool = False, filename:str = "NA"
        ):
        """Set the properties of the table under test. Call it before inittest().

        Args:
            tablename: Name used in logs and reports (spaces are removed).
            coltest: Run the column test in comparecolumns().
            coldatatypetest: In the column test, also compare data types.
            datatest: Marks key/data tests as enabled in the sample and HTML reports.
            savetable: Currently not used.
            debug: Log counts, schemas and samples (adds Spark actions).
            cleanpklevel: Level passed to pk() when building keys (see Atom_UDF.pk).
            repartition: Currently not used.
            excolsincoltest: Columns excluded from the column test (use lowercase names).
            excolsindatatest: Columns excluded from the data test (use lowercase names).
            reversecheck: In comparedata(), report matching values instead of mismatches.
            filename: Value stored as FILE_NAME in the run summary.

        input_filename is always added to both exclusion lists.
        """
        self.__tableprop_status=True
        tablename = tablename.replace(" ", "")

        self.__src_trg_file_list = []
        if input_filename not in excolsincoltest:
            excolsincoltest.append(input_filename)

        if input_filename not in excolsindatatest:
            excolsindatatest.append(input_filename)

        self.__tableprops = {
            "TABLENAME": tablename, "COLTEST": coltest, "COLTEST-DATATYPE": coldatatypetest,
            "DATATEST": datatest, "SAVETABLE": savetable, "DEBUG": debug, "CLEANPKLEVEL": cleanpklevel,
            "REPARTITION": repartition, "EXCLUDECOLS-COLTEST": excolsincoltest,
            "EXCLUDECOLS-DATATEST": excolsindatatest, "REVERSECHK": reversecheck,
            "FILE_NAME": filename
        }
        return self.__tableprops

    def inittest(self):
        """Start the test for the current table: check the project info (name >= 3, description >= 10
        characters) and reset the run and table summaries.

        Load and compare methods only run after settableproperty() and a successful inittest().
        """
        cond = len(self._settings.projectname)>=3 and len(self._settings.projectdesc)>=10
        self.__test_init = cond

        self.__src_trg_file_list = []
        self.__ressummary = {
            'FILE_NAME': self.__tableprops["FILE_NAME"], 'TABLE_COUNT': 0,
            'START_TIME': self.__curdate, 'END_TIME': '',
            'MD_SRC_COL_COUNT': 0, 'MD_TRG_COL_COUNT': 0,
            'COLTEST_ERRORS': 0,   'KEYTEST_ERRORS': 0,
            'DT_SRC_COL_COUNT': 0, 'DT_TRG_COL_COUNT': 0,
            'SRC_REC_COUNT': 0,    'TRG_REC_COUNT': 0,
            'DATATEST_ERRORS': 0,
            'DATAQUALITY_TESTS': -1,  'DATAQUALITY_TEST_PASSED': -1,
            'DATAQUALITY_ERRORS': -1, 'DATAQUALITY_TEST_FAILED': -1
        }

        default_dict = {'START_STATUS': False, 'END_STATUS': False, 'START_TIME': '', 'END_TIME': '', 'EXCEPTION': ''}
        self.__resdetailsummary = {
            'TABLE_NAME': self.__tableprops["TABLENAME"],
            'START_TIME': Atom_UDF.gettimestamp(),
            'END_TIME': '',
            'COL_TEST': {
                'TEST_TYPE':'','SRC_COL_COUNT': 0, 'TRG_COL_COUNT': 0,
                'COLTEST_ERRORS': 0, 'CUR_COLTEST_ERRORS': 0, **default_dict
            },
            'KEY_TEST': {'MATCH_FOUND': 0,'KEYTEST_ERRORS': 0, **default_dict},
            'DATA_TEST': {
                'SRC_COL_COUNT': 0, 'SRC_REC_COUNT': 0,
                'TRG_COL_COUNT': 0, 'TRG_REC_COUNT': 0,
                'FAILED_COLS': 0, 'DATATEST_ERRORS': 0, **default_dict
            },
            'DATAQUALITY': {}
        }
        return
    
    # select_columns - list of columns to be viewed when returning df
    # stop_at_column - column name to stop flattening at
    def flatten(self, df, select_columns:List[str] = [], stop_at_column:str = "") -> DataFrame:
        """Flatten nested columns, pass by pass, until none remain.

        Each pass: struct fields become parent_child columns, arrays are exploded with explode_outer
        (one row per element; null/empty arrays keep one row) and maps are pivoted into one column per key.

        Args:
            select_columns: Columns to return. Flattening stops early once all of them exist.
            stop_at_column: Stop and return the current DataFrame when this column is found.

        The temp view of the loaded data is not updated with the flattened result.
        """
        while True:
            schema = df.schema
            struct_cols = []
            array_cols = []
            map_cols = []
            for field in schema.fields:
                if stop_at_column and field.name == stop_at_column:
                    return df.select(*select_columns) if select_columns else df

                if isinstance(field.dataType, T.StructType):
                    struct_cols.append(field)
                elif isinstance(field.dataType, T.ArrayType):
                    array_cols.append(field.name)
                elif isinstance(field.dataType, MapType):
                    map_cols.append(field.name)

            # Nothing left to flatten
            if not struct_cols and not array_cols and not map_cols:
                break

            # ---- Flatten all structs in ONE projection ----
            if struct_cols:
                new_cols  = []
                drop_cols = []
                for field in struct_cols:
                    for sub in field.dataType.fields:
                        new_col = F.col(f"`{field.name}`.`{sub.name}`").alias(f"{field.name}_{sub.name.replace('.', '_')}")
                        new_cols.append(new_col)
                    drop_cols.append(field.name)

                df = df.select("*", *new_cols).drop(*drop_cols)

            for col_name in array_cols:# Explode arrays
                df = df.withColumn(col_name, F.explode_outer(col_name))

            # ---- Handle maps efficiently (explode → pivot once) ----
            for col_name in map_cols:
                df = df.select(
                    "*", F.explode(col_name).alias(f"{col_name}_key", f"{col_name}_value")
                ).groupBy(
                    [c for c in df.columns if c not in {f"{col_name}_key", f"{col_name}_value"}]
                ).pivot(f"{col_name}_key").agg(F.first(f"{col_name}_value")).drop(col_name)

            # ---- Early exit ----
            if select_columns and all(c in df.columns for c in select_columns):
                return df.select(*select_columns)
        return df.select(*select_columns) if select_columns else df

    # flatten structs
    def __flatten_df(self, df:DataFrame, prefix:Optional[str] = None):
        """Creates a query to flatten df"""
        def get_field_name(alias:str, field:str) -> str:
            alias = alias.replace(".", "_")
            alias = Atom_UDF.fieldname(alias)
            return f"{field} as `{alias}`"
        
        flatten_required = False
        fields = []
        df_schema = df.schema
        for field in df_schema.fields:
            dtype = field.dataType

            if isinstance(dtype, MapType):
                flatten_required = True
                keys_df = df.select(F.explode(F.map_keys(F.col(field.name)))).distinct()
                keys = list(map(lambda row: row[0], keys_df.collect()))
                for key in keys:
                    alias = f"{field.name}_{key.strip()}" if prefix is None else f"{prefix}_{field.name}_{key.strip()}"
                    field_name = get_field_name(alias, f"{field.name}['`{key}`']")
                    fields.append(field_name)

            elif isinstance(dtype, T.StructType):
                flatten_required = True
                for k in dtype.fields:
                    field_name = get_field_name(f"{field.name}_{k.name}", f"{field.name}.`{k.name}`")
                    fields.append(field_name)
            else:
                alias = field.name if prefix is None else f"{prefix}_{field.name}"
                field_name = get_field_name(Atom_UDF.fieldname(alias), field.name)
                fields.append(field_name)

        return {"RES": flatten_required, "COLS": ", ".join(fields)}

    # flatten array and structs
    def __explodedataframe(self, explodestruct:bool, explodearray:bool, df:DataFrame, tempview:str, requiredcols:Optional[list] = None):
        """Expand *df* step by step and replace the temp view *tempview* after each step.

        explodestruct: expand struct/map columns one level per step (see __flatten_df).
        explodearray: explode the first array column per step with explode_outer.
        Column names are sanitized with Atom_UDF.fieldname(). Stops when nothing is left to expand
        or when all *requiredcols* exist.
        """
        requiredcols = requiredcols or []

        required_cols_found = lambda: requiredcols and all(c in df.columns for c in requiredcols)
        while True:
            if explodestruct:
                if required_cols_found():
                    break
                
                res = self.__flatten_df(df)
                if not res["RES"]:
                    break

                df = df.selectExpr(*res["COLS"].split(","))
                self._settings.createTempViews(df, tempview)

            if explodearray:
                while True:
                    if required_cols_found():
                        break

                    array_found = False
                    select_cols = []
                    condition = lambda dtype: isinstance(dtype, T.ArrayType) and not array_found
                    for s in df.schema.fields:
                        aliasname = Atom_UDF.fieldname(f"{s.name}")
                        if condition(s.dataType):
                            array_found = True
                            select_cols.append(f"explode_outer(`{s.name}`) AS {aliasname}")
                        else:
                            select_cols.append(f"`{s.name}` AS {aliasname}")
                            
                    if array_found:
                        df = df.selectExpr(*select_cols)
                        self._settings.createTempViews(df, tempview)
                    else:
                        break

                if not explodestruct and not array_found:
                    break
        return df

    def __check_run(self):
        """True when settableproperty() and inittest() have both succeeded."""
        return   self.__tableprop_status and self.__test_init

    def __normalize_dataset(
        self,
        dataset: DataFrame,
        normalize: Optional[NormalizeMode],
        normalize_cols: Optional[List[str]] = None,
        normalize_exclude: Optional[List[str]] = None,
        normalize_tag_separator: str = "",
        normalize_decode_entities: bool = True,
        normalize_transliterate: bool = True,
    ) -> DataFrame:
        """Normalize the columns of a loaded dataset when *normalize* is set; otherwise return it unchanged.

        Used by every load* method, before explode/flatten (strings inside structs are not normalized).

        Args:
            normalize: None (default) = no change. "applicable" = string columns, "all" = every
                       non-complex column (cast to string), "selected" = only *normalize_cols*.
            normalize_cols: Columns for mode "selected" (case-insensitive).
            normalize_exclude: Columns never normalized; input_filename is always excluded.
            normalize_tag_separator: Text that replaces inner HTML tags ("" removes them).
            normalize_decode_entities: Decode HTML entities ("&amp;" -> "&"). False keeps them literally.
            normalize_transliterate: Transliterate to ASCII (accents removed). False keeps accents and symbols.
        """
        if not normalize:
            return dataset
        # The technical file-name column is never normalized.
        exclude = [input_filename, *(normalize_exclude or [])]
        Writer.logger(
            f"normalizing columns (mode={normalize}, tag_separator='{normalize_tag_separator}', "
            f"decode_entities={normalize_decode_entities}, transliterate={normalize_transliterate})...", True
        )
        return Atom_UDF.normalize_columns(
            dataset, mode=normalize, columns=normalize_cols, exclude=exclude,
            tag_separator=normalize_tag_separator,
            decode_entities=normalize_decode_entities,
            transliterate=normalize_transliterate,
        )

    #Debugging function to log schema, samples and record count of dataframe
    def debug(self, message:str, dataset:DataFrame, tempview:str):
        """Log *message*, the view name, the row count, the schema and one sample row of *dataset* (runs Spark actions)."""
        Writer.logger(message)
        Writer.logger(f"Temp view created as {tempview}")
        Writer.logger(f"Records => {dataset.count()}")
        Writer.logger(self.__getDFSchema(dataset))
        Writer.logger(self.__getDFSamples(dataset))
        return

    def __load_single_excel(self, file_name:str, sheet_name:Optional[str], header:Optional[bool]) -> DataFrame:
        """
            Reads a single Excel file from a supported path and returns a Spark DataFrame.

            Args:
                file_name(str): Full path to the Excel file.
                header(bool, optional): True uses the first row as column names; None/False names the columns 0..n-1.
                sheet_name(str, optional): Sheet name to read. If not provided, the first sheet is used.

            Returns:
                (DataFrame): Spark DataFrame containing the Excel data.
        """
        spark = SparkSession.builder.getOrCreate()

        binary_df = spark.read.format("binaryFile").load(file_name)
        content = binary_df.select("content").first()["content"]

        wb = load_workbook(BytesIO(content), read_only=True)

        sheet_name = sheet_name or wb.sheetnames[0]

        ws = wb[sheet_name]

        data = list(ws.values)

        if not data:
            raise ValueError(f"No data found in sheet '{sheet_name}'")

        if header:
            columns = data[0]
            pdf = pd.DataFrame(data[1:], columns=columns)
        else:
            pdf = pd.DataFrame(data)
        
        for col in pdf.columns:
            pdf[col] = pdf[col].astype("string")

        return spark.createDataFrame(pdf)

    def load_excel_files(self, obj_list:List[Dict[str,str]], view_name:str, normalize:Optional[NormalizeMode] = None, normalize_cols:Optional[List[str]] = None, normalize_exclude:Optional[List[str]] = None, normalize_tag_separator:str = "", normalize_decode_entities:bool = True, normalize_transliterate:bool = True) -> Optional[DataFrame]:
        """
        Reads multiple Excel files and returns a single Spark DataFrame.

        Args:
            obj_list (List[Dict[str, str]]): List of dictionaries with the following structure:
                {
                    "file_name": "<path_to_file>",
                    "sheet_name": "<sheet_name_or_None>",
                    "header": <True or False> # if None, it will be set to False
                }
            view_name (str): Temp view name to be created.
            normalize*: See __normalize_dataset.

        Returns
            (DataFrame): Union by name of all loaded Excel files (missing columns are null),
            or None if the test is not initiated or the union fails.
        """
        if not self.__check_run():
            self.__clear_excel_obj()
            print(TEST_NOT_INITIATED_MESSAGE)
            return None
        
        if not obj_list:
            Writer.logger("No Excel files were provided", True)
            return None
        
        Writer.logger(f"load excel file...", True)
        spark = SparkSession.builder.getOrCreate()

        dfs = []
        for dct in obj_list:
            try:
                tmp = self.__load_single_excel(**dct)
            except Exception as e:
                Writer.logger(f"Could not read xlsx file: {dct.get('file_path')}")
                pass
            dfs.append(tmp)
        
        try:
            df = dfs[0]
            for df_i in dfs[1:]:
                df = df.unionByName(df_i, allowMissingColumns = True)
        except Exception as e:
            Writer.logger(f"exception in loadexcel\n{e}", True)
            return None
        
        df = self.__normalize_dataset(df, normalize, normalize_cols, normalize_exclude, normalize_tag_separator, normalize_decode_entities, normalize_transliterate)

        if self.__tableprops.get("DEBUG"):
            df.cache()
        
        self._settings.createTempViews(df, view_name)
        if view_name not in self.__src_trg_file_list:
            self.__src_trg_file_list.append(view_name)

        if self.__tableprops.get("DEBUG"):
            self.debug(f"Read excel file", df, view_name)
        return df

    #load excel
    def load_excel(self, path:str, view_name:str, sheet_name:Optional[str] = None, header:Optional[bool] = None, normalize:Optional[NormalizeMode] = None, normalize_cols:Optional[List[str]] = None, normalize_exclude:Optional[List[str]] = None, normalize_tag_separator:str = "", normalize_decode_entities:bool = True, normalize_transliterate:bool = True) -> Optional[DataFrame]:
        """Read one sheet of an Excel file (all values as strings) into the temp view *view_name*.

        Args:
            path: Excel file path (read with binaryFile; the whole file is loaded in driver memory).
            sheet_name: Sheet to read (default: the first sheet).
            header: True uses the first row as column names; otherwise the columns are named 0..n-1.
            normalize*: See __normalize_dataset.
        """
        if not self.__check_run():
            print(TEST_NOT_INITIATED_MESSAGE)
            return None
        
        Writer.logger(f"load excel file...", True)
        spark = SparkSession.builder.getOrCreate()
        try:
            df = self.__load_single_excel(path, sheet_name, header)
        except Exception as e:
            Writer.logger(f"exception in loadexcel\n{e}", True)
        
        df = self.__normalize_dataset(df, normalize, normalize_cols, normalize_exclude, normalize_tag_separator, normalize_decode_entities, normalize_transliterate)

        if self.__tableprops.get("DEBUG"):
            df.cache()
        
        self._settings.createTempViews(df, view_name)
        if view_name not in self.__src_trg_file_list:
            self.__src_trg_file_list.append(view_name)

        if self.__tableprops.get("DEBUG"):
            self.debug(f"Read excel file", df, view_name)
        return df
        
    # load csv file
    def loadcsv(self, filename:str, tempview:str, useroptions:dict = {}, normalize:Optional[NormalizeMode] = None, normalize_cols:Optional[List[str]] = None, normalize_exclude:Optional[List[str]] = None, normalize_tag_separator:str = "", normalize_decode_entities:bool = True, normalize_transliterate:bool = True):
        """Read CSV file(s) into the temp view *tempview* and add the column input_filename (source file path).

        Default options: header, multiline, quote '"', escape backslash, delimiter ","; *useroptions* override them.
        Line breaks in the last header name are removed.
        normalize*: See __normalize_dataset.

        Returns the DataFrame, or None on error or if the test is not initiated.
        """
        if not self.__check_run():
            print(TEST_NOT_INITIATED_MESSAGE)
            return None

        msg = "[loadcsv] Using User provided CSV options" if useroptions else "[loadcsv] Using default CSV options"
        success = True
        try:
            Writer.logger("load csv files...", True)
            csvdefaultoptions = {
                "header": "true", "wholeFile": "true", "multiline": "true",
                "quote": "\"", "escape": "\\", "delimiter": ","
            }
            useroptions = self.__update_dict(useroptions, csvdefaultoptions)

            print(msg)
            dataset = self._settings.sprk.read.format("csv").options(**useroptions).load(filename) \
                .withColumn(input_filename, F.col("_metadata.file_path"))

            print("[loadcsv] Dataset read completed")
            dataset.cache()
            lastcolumnname = dataset.columns[-2]

            if '\r' in lastcolumnname or '\n' in lastcolumnname:
                newcolumnname = Atom_UDF.cleanNewLine(lastcolumnname)
                dataset = dataset.withColumnRenamed(lastcolumnname, newcolumnname)

            dataset = self.__normalize_dataset(dataset, normalize, normalize_cols, normalize_exclude, normalize_tag_separator, normalize_decode_entities, normalize_transliterate)
            self._settings.createTempViews(dataset, tempview)
            if tempview not in self.__src_trg_file_list:
                self.__src_trg_file_list.append(tempview)
            # self.__explodedataframe(dataset, tempview)

            if self.__tableprops.get("DEBUG"):
                self.debug(f"Read csv file from {filename}", dataset, tempview)
            print("returning dataset from loadcsv")

        except Exception as e:
            Writer.logger("exception in loadcsv")
            Writer.logger(e)
            success = False
        return dataset if success else None

    # load orc file
    def loadorc(
            self, filename:str, tempview:str, useroptions:dict = {}, explodestruct:bool = False,
            explodearray:bool = False, flatten:bool = False, select_columns:list = None, stop_at_column:str = None,
            normalize:Optional[NormalizeMode] = None, normalize_cols:Optional[List[str]] = None, normalize_exclude:Optional[List[str]] = None,
            normalize_tag_separator:str = "",
            normalize_decode_entities:bool = True, normalize_transliterate:bool = True
        ):
        """Read ORC file(s) into the temp view *tempview* and add the column input_filename.

        Args:
            useroptions: Reader options.
            explodestruct / explodearray: Expand structs / explode arrays step by step (see __explodedataframe).
            flatten, select_columns, stop_at_column: Flatten the result with flatten().
            normalize*: See __normalize_dataset (applied before explode/flatten).

        Returns the DataFrame, or None on error or if the test is not initiated.
        """
        if not self.__check_run():
            print(TEST_NOT_INITIATED_MESSAGE)
            return None
        
        success = True
        try:
            Writer.logger("load orc files...", True)

            reader = self._settings.sprk.read.format("orc")
            if useroptions:
                reader = reader.options(**useroptions)
            dataset = reader.load(filename).withColumn(input_filename, F.col("_metadata.file_path"))
            dataset = self.__normalize_dataset(dataset, normalize, normalize_cols, normalize_exclude, normalize_tag_separator, normalize_decode_entities, normalize_transliterate)

            # dataset.createOrReplaceTempView(tempview)
            if self.__tableprops.get("DEBUG"):
                dataset.cache()
            self._settings.createTempViews(dataset, tempview)

            if tempview not in self.__src_trg_file_list:
                self.__src_trg_file_list.append(tempview)

            if explodestruct or explodearray:
                dataset = self.__explodedataframe(explodestruct, explodearray, dataset, tempview)

            if flatten:
                dataset = self.flatten(dataset, select_columns, stop_at_column)

            if self.__tableprops.get("DEBUG"):
                self.debug(f"Read orc file from {filename}", dataset, tempview)
            
        except Exception as e:
            Writer.logger("exception in loadorc")
            Writer.logger(e)
            success = False
        return dataset if success else None

    # load xml file
    def loadxml(
            self, filename:str, tempview:str, useroptions:dict = {}, explodestruct:bool = False,
            explodearray:bool = False, flatten:bool = False, select_columns:list = None, stop_at_column:str = None,
            normalize:Optional[NormalizeMode] = None, normalize_cols:Optional[List[str]] = None, normalize_exclude:Optional[List[str]] = None,
            normalize_tag_separator:str = "",
            normalize_decode_entities:bool = True, normalize_transliterate:bool = True
        ):
        """Read XML file(s) into the temp view *tempview* and add the column input_filename.

        Set rowTag (and optionally rootTag) in *useroptions*; their defaults are empty.
        explode*/flatten/normalize*: as in loadorc().

        Returns the DataFrame, or None on error or if the test is not initiated.
        """
        if not self.__check_run():
            print(TEST_NOT_INITIATED_MESSAGE)
            return None
        
        success = True
        try:
            Writer.logger("load xml files...", True)
            useroptions = self.__update_dict(useroptions, {"rootTag": "", "rowTag": "", "wholeFile": "true"})
            dataset = self._settings.sprk.read.format("databricks.spark.xml").options(**useroptions).load(filename) \
                .withColumn(input_filename, F.col("_metadata.file_path"))
            dataset = self.__normalize_dataset(dataset, normalize, normalize_cols, normalize_exclude, normalize_tag_separator, normalize_decode_entities, normalize_transliterate)

            if self.__tableprops.get("DEBUG"):
                dataset.cache()
            self._settings.createTempViews(dataset, tempview)

            if tempview not in self.__src_trg_file_list:
                self.__src_trg_file_list.append(tempview)

            if explodestruct or explodearray:
                dataset = self.__explodedataframe(explodestruct, explodearray, dataset, tempview)

            if flatten:
                dataset = self.flatten(dataset, select_columns, stop_at_column)

            if self.__tableprops.get("DEBUG"):
                self.debug(f"Read xml file from {filename}", dataset, tempview)
        
        except Exception as e:
            Writer.logger(f"Exception in loadxml {e}")
            success = False
        return dataset if success else None

    # load json file
    def loadjson(
            self, filename:str, tempview:str, useroptions:dict = {}, explodestruct:bool = False,
            explodearray:bool = False, flatten:bool = False, select_columns:list = [], stop_at_column:str = None,
            normalize:Optional[NormalizeMode] = None, normalize_cols:Optional[List[str]] = None, normalize_exclude:Optional[List[str]] = None,
            normalize_tag_separator:str = "", jsonl:Optional[bool] = None,
            normalize_decode_entities:bool = True, normalize_transliterate:bool = True
        ):
        """Load JSON or JSON Lines (JSONL / NDJSON) files into a temp view.

        Args:
            jsonl: None  -> auto-detect from the file extension (.jsonl / .ndjson, optionally compressed).
                   True  -> force JSON Lines mode (one JSON object per line).
                   False -> force standard multi-line JSON mode (one document per file).
            useroptions: Reader options.
            explode*/flatten/normalize*: as in loadorc().

        Adds the column input_filename. Returns the DataFrame, or None on error or if the test is not initiated.
        """
        if not self.__check_run():
            print(TEST_NOT_INITIATED_MESSAGE)
            return None
        
        success = True
        try:
            Writer.logger("load json files...", True)
            options = self.__update_dict(dict(useroptions or {}), {"multiline": "false"})

            is_jsonl = _is_jsonl_path(filename) if jsonl is None else jsonl
            if is_jsonl:
                options["multiline"] = "false"  # JSON Lines: one record per line
                if jsonl is True and not _is_jsonl_path(filename) and "pathGlobFilter" not in options:
                    options["pathGlobFilter"] = "*.jsonl*"  # Directory: read only .jsonl files
            elif jsonl is False:
                options["multiline"] = "true"   # Standard JSON: one document per file
            Writer.logger(f"[loadjson] format={'jsonl' if is_jsonl else 'json'}, multiline={options['multiline']}", True)

            dataset = self._settings.sprk.read.format("json").options(**options).load(filename) \
                .withColumn(input_filename, F.col("_metadata.file_path"))
            dataset = self.__normalize_dataset(dataset, normalize, normalize_cols, normalize_exclude, normalize_tag_separator, normalize_decode_entities, normalize_transliterate)

            if self.__tableprops.get("DEBUG"):
                dataset.cache()
            
            self._settings.createTempViews(dataset, tempview)
            if tempview not in self.__src_trg_file_list:
                self.__src_trg_file_list.append(tempview)

            if explodestruct or explodearray:
                dataset = self.__explodedataframe(explodestruct, explodearray, dataset, tempview)

            if flatten:
                dataset = self.flatten(dataset, select_columns, stop_at_column)

            if self.__tableprops.get("DEBUG"):
                self.debug(f"Read json file from {filename}", dataset, tempview)

        except Exception as e:
            Writer.logger("exception in loadjson", True)
            Writer.logger(e, True)
            success = False
        return dataset if success else None

    # load parquet file
    def loadparquet(
            self, filename:str, tempview:str, useroptions:dict = {}, explodestruct:bool = False,
            explodearray:bool = False, flatten:bool = False, select_columns:list = [], stop_at_column:str = None,
            normalize:Optional[NormalizeMode] = None, normalize_cols:Optional[List[str]] = None, normalize_exclude:Optional[List[str]] = None,
            normalize_tag_separator:str = "",
            normalize_decode_entities:bool = True, normalize_transliterate:bool = True
        ):
        """Read Parquet file(s) into the temp view *tempview* and add the column input_filename.

        useroptions/explode*/flatten/normalize*: as in loadorc().
        Returns the DataFrame, or None on error or if the test is not initiated.
        """
        if not self.__check_run():
            print(TEST_NOT_INITIATED_MESSAGE)
            return None
        
        success = True
        try:
            Writer.logger("load parquet files...", True)
            reader = self._settings.sprk.read.format("parquet")
            if useroptions:
                reader = reader.options(**useroptions)
            
            dataset = reader.load(filename).withColumn(input_filename, F.col("_metadata.file_path"))
            dataset = self.__normalize_dataset(dataset, normalize, normalize_cols, normalize_exclude, normalize_tag_separator, normalize_decode_entities, normalize_transliterate)

            if self.__tableprops.get("DEBUG"):
                dataset.cache()
            self._settings.createTempViews(dataset, tempview)

            if tempview not in self.__src_trg_file_list:
                self.__src_trg_file_list.append(tempview)

            if explodestruct or explodearray:
                dataset = self.__explodedataframe(explodestruct, explodearray, dataset, tempview)

            if flatten:
                dataset = self.flatten(dataset, select_columns, stop_at_column)

            if self.__tableprops.get("DEBUG"):
                self.debug(f"Read parquet file from {filename}", dataset, tempview)
            
        except Exception as e:
            Writer.logger("exception in loadparquet")
            Writer.logger(e)
            success = False
        return dataset if success else None
        
    # load delta file
    def loaddelta(
            self, filename:str, tempview:str, useroptions:dict = {}, explodestruct:bool = False, explodearray:bool = False,
            flatten:bool = False, select_columns:list = [], stop_at_column:str = None,
            normalize:Optional[NormalizeMode] = None, normalize_cols:Optional[List[str]] = None, normalize_exclude:Optional[List[str]] = None,
            normalize_tag_separator:str = "",
            normalize_decode_entities:bool = True, normalize_transliterate:bool = True
        ):
        """Read a Delta table by path into the temp view *tempview* and add the column input_filename.

        useroptions/explode*/flatten/normalize*: as in loadorc().
        Returns the DataFrame, or None on error or if the test is not initiated.
        """
        if not self.__check_run():
            print(TEST_NOT_INITIATED_MESSAGE)
            return None
        
        success = True
        try:
            Writer.logger("load delta files...", True)
            reader = self._settings.sprk.read.format("delta")
            if useroptions:
                reader = reader.options(**useroptions)
            dataset = reader.load(filename).withColumn(input_filename, F.col("_metadata.file_path"))
            dataset = self.__normalize_dataset(dataset, normalize, normalize_cols, normalize_exclude, normalize_tag_separator, normalize_decode_entities, normalize_transliterate)

            if self.__tableprops.get("DEBUG"):
                dataset.cache()
            self._settings.createTempViews(dataset, tempview)

            if tempview not in self.__src_trg_file_list:
                self.__src_trg_file_list.append(tempview)

            if explodestruct or explodearray:
                dataset = self.__explodedataframe(explodestruct, explodearray, dataset, tempview)

            if flatten:
                dataset = self.flatten(dataset, select_columns, stop_at_column)

            if self.__tableprops.get("DEBUG"):
                self.debug(f"Read delta file from {filename}", dataset, tempview)

        except Exception as e:
            Writer.logger("exception in loaddelta")
            Writer.logger(e)
            success = False
        return dataset if success else None
        
    # load hive table
    def loadhive(
            self, hivetable:str, tempview:str, explodestruct:bool = False, explodearray:bool = False,
            flatten:bool = False, select_columns:list = [], stop_at_column:str = None,
            normalize:Optional[NormalizeMode] = None, normalize_cols:Optional[List[str]] = None, normalize_exclude:Optional[List[str]] = None,
            normalize_tag_separator:str = "",
            normalize_decode_entities:bool = True, normalize_transliterate:bool = True
        ):
        """Load a table or a SQL query into the temp view *tempview*.

        *hivetable* is run as SQL when it contains select/from/with, a space or a line break;
        otherwise it is read with "select * from <hivetable>". No input_filename column is added.
        explode*/flatten/normalize*: as in loadorc().

        Returns the DataFrame, or None on error or if the test is not initiated.
        """
        if not self.__check_run():
            print(TEST_NOT_INITIATED_MESSAGE)
            return None
        
        success = True
        try:
            Writer.logger("load hive tables...", True)
            is_sql = any(x in hivetable for x in ('select', 'from', 'with', ' ', '\r', '\n'))
            dataset = self._settings.sprk.sql(hivetable if is_sql else f"select * from {hivetable}")
            dataset = self.__normalize_dataset(dataset, normalize, normalize_cols, normalize_exclude, normalize_tag_separator, normalize_decode_entities, normalize_transliterate)

            if self.__tableprops.get("DEBUG"):
                dataset.cache()
            self._settings.createTempViews(dataset, tempview)

            if tempview not in self.__src_trg_file_list:
                self.__src_trg_file_list.append(tempview)

            if explodestruct or explodearray:
                dataset = self.__explodedataframe(explodestruct, explodearray, dataset, tempview)

            if flatten:
                dataset = self.flatten(dataset, select_columns, stop_at_column)

            if self.__tableprops.get("DEBUG"):
                self.debug(f"Read hive table from {hivetable}", dataset, tempview)

        except Exception as e:
            Writer.logger("exception in loadhive")
            Writer.logger(e)
            success = False
        return dataset if success else None

    # load data from any jdbc
    #for oracle use useroptions={"query":"select * from table"}
    #select * from table where column='ddd' --> with tmp_ora as (select * from table where column='ddd') --> select * from tmp_ora
    #with tmp_ora as (select * from table where column='ddd') -> useroptions={"query":"select * from tmp_ora"}
    def loadrdbms(
            self, driver:str, jdbcuri:str, dbuser:str, dbpassword:str, dbtable:str, tempview:str,
            useroptions:dict = {}, explodestruct:bool = False, explodearray:bool = False, flatten:bool = False,
            select_columns:list = [], stop_at_column:str = None,
            normalize:Optional[NormalizeMode] = None, normalize_cols:Optional[List[str]] = None, normalize_exclude:Optional[List[str]] = None,
            normalize_tag_separator:str = "",
            normalize_decode_entities:bool = True, normalize_transliterate:bool = True
        ):
        """Read a JDBC table or query into the temp view *tempview*.

        Args:
            driver, jdbcuri, dbuser: JDBC connection settings.
            dbpassword: Decrypted with PasswordManager.decrypt() when Settings.vencrypted_password is truthy.
            dbtable: Table name (no spaces) or a query. Oracle: the query is wrapped as
                     "with tmp_ora as (...) select * from tmp_ora"; if it starts with WITH,
                     useroptions["query"] is required. Other drivers: the query is sent as prepareQuery
                     with useroptions["query"] (default "select 1").
            useroptions: Extra JDBC options (they override the connection settings).
            explode*/flatten/normalize*: as in loadorc().

        No input_filename column is added. Returns the DataFrame, or None on error or if the test is not initiated.
        """
        if not self.__check_run():
            print(TEST_NOT_INITIATED_MESSAGE)
            return None
        
        success = True
        try:
            Writer.logger("load rdbms tables...", True)
            vdbpass = PasswordManager.decrypt(dbpassword) if self._settings.vencrypted_password else dbpassword

            # Check if exists in dict: dbtable or (prepareQuery and query) 
            rdbmsdefaultoptions = {"url": jdbcuri, "user": dbuser, "password": vdbpass, "driver": driver}
            if " " not in str(dbtable).strip():
                rdbmsdefaultoptions["dbtable"] = dbtable
            else:
                if "oracle.jdbc.driver.oracledriver" in str(driver).lower():# for oracle
                    if str(dbtable).lower().startswith("with"):
                        prm_qry = useroptions.get("query", None)
                        if prm_qry is not None:
                            rdbmsdefaultoptions = self.__update_dict(rdbmsdefaultoptions, {"prepareQuery": dbtable, "query": prm_qry})
                        else:
                            Writer.logger("missing query in user options, set useroptions={'query':'sqlquery'}", True)
                    else:
                        rdbmsdefaultoptions = self.__update_dict(rdbmsdefaultoptions, {
                            "prepareQuery": f"with tmp_ora as ({dbtable})", "query": "select * from tmp_ora"
                        })
                else:# for other than oracle
                    rdbmsdefaultoptions = {
                        "url": jdbcuri, "prepareQuery": dbtable, "user": dbuser,
                        "password": vdbpass, "driver": driver, "query": useroptions.get("query", "select 1")
                    }

            if not ("dbtable" in rdbmsdefaultoptions or "query" in rdbmsdefaultoptions):
                raise Exception("missing mandatory options for rdbms connection, check your options and try again")
            useroptions = self.__update_dict(useroptions, rdbmsdefaultoptions)
            dataset = self._settings.sprk.read.format("jdbc").options(**useroptions).load()
            dataset = self.__normalize_dataset(dataset, normalize, normalize_cols, normalize_exclude, normalize_tag_separator, normalize_decode_entities, normalize_transliterate)

            if self.__tableprops.get("DEBUG"):
                dataset.cache()

            self._settings.createTempViews(dataset, tempview)
            if tempview not in self.__src_trg_file_list:
                self.__src_trg_file_list.append(tempview)
            if explodestruct or explodearray:
                dataset = self.__explodedataframe(explodestruct, explodearray, dataset, tempview)
            if flatten:
                dataset = self.flatten(dataset, select_columns, stop_at_column)
            if self.__tableprops.get("DEBUG"):
                self.debug(f"Read rdbms from {jdbcuri}", dataset, tempview)

        except Exception as e:
            Writer.logger("exception in loadrdbms")
            Writer.logger(e)
            success = False
        return dataset if success else None

    # clear response collection
    def clearResponsecollection(self):
        """Clear the stored REST responses used to resolve variables in loadrestapi()."""
        self.__responsecollection = {}

    # load data from rest api
    def loadrestapi(
            self, method:str = "GET", endpoint:str = "", headers:dict = {}, data:dict = {}, jsondata:dict = {}, variables:dict = {},
            timeout:int = 10000, retry:int = 1, tempview:str = "", sql:str = "", ignoressl:bool = True, explodestruct:bool = False,
            explodearray:bool = False, flatten:bool = False, select_columns:list = None, stop_at_column:str = None,
            normalize:Optional[NormalizeMode] = None, normalize_cols:Optional[List[str]] = None, normalize_exclude:Optional[List[str]] = None,
            normalize_tag_separator:str = "",
            normalize_decode_entities:bool = True, normalize_transliterate:bool = True
        ):
        """Call a REST API and store the JSON response under *tempview* in the response collection
        (used to resolve *variables* in later calls).

        When *sql* is given, the response is exposed as the temp view *tempview*, *sql* is run on it,
        its result replaces the view and the response collection is cleared.
        explode*/flatten/normalize*: as in loadorc(), only applied when *sql* is given.

        Returns the *sql* result DataFrame, the raw JSON response when *sql* is empty,
        or None on error or if the test is not initiated.
        """

        if not self.__check_run():
            print(TEST_NOT_INITIATED_MESSAGE)
            return None
        
        success = True
        try:
            Writer.logger("load restapi...", True)
            #defHeaders = {"Content-Type": "application/json", "Accept": "application/json"}
            #headers = self.__update_dict(headers, defHeaders)
            dataset = None
            r = Rest()
            r.set_parameters("restapi", endpoint, method, data, jsondata, variables, self.__responsecollection, headers, timeout, retry, ignoressl)
            jsonresponse = r.getresponse()
            self.__responsecollection[tempview] = jsonresponse

            if sql.strip():
                # r.getdataframe(jsonresponse).createOrReplaceTempView(tempview)
                self._settings.createTempViews(r.getdataframe(jsonresponse), tempview)
                dataset = self._settings.sprk.sql(sql)
                dataset = self.__normalize_dataset(dataset, normalize, normalize_cols, normalize_exclude, normalize_tag_separator, normalize_decode_entities, normalize_transliterate)
                
                if self.__tableprops.get("DEBUG"):
                    dataset.cache()
                
                self._settings.createTempViews(dataset, tempview)
                if tempview not in self.__src_trg_file_list:
                    self.__src_trg_file_list.append(tempview)

                if explodestruct or explodearray:
                    dataset = self.__explodedataframe(explodestruct, explodearray, dataset, tempview)

                if flatten:
                    dataset = self.flatten(dataset, select_columns, stop_at_column)

                self.__responsecollection = {}

        except Exception as e:
            Writer.logger("exception in loadrestapi")
            Writer.logger(e)
            success = False
        return None if not success else (dataset if sql.strip() else jsonresponse)

    # load data from static sql
    def loadsql(self, sql, tempview, normalize:Optional[NormalizeMode] = None, normalize_cols:Optional[List[str]] = None, normalize_exclude:Optional[List[str]] = None, normalize_tag_separator:str = "", normalize_decode_entities:bool = True, normalize_transliterate:bool = True) -> Union[DataFrame, None]:
        """Run *sql* and register the result as the temp view *tempview* (normalize*: see __normalize_dataset).

        Returns the DataFrame, or None on error or if the test is not initiated.
        """
        if not self.__check_run():
            print(TEST_NOT_INITIATED_MESSAGE)
            return None
        
        success = True
        try:
            Writer.logger("load sql...", True)
            dataset = self._settings.sprk.sql(sql)
            dataset = self.__normalize_dataset(dataset, normalize, normalize_cols, normalize_exclude, normalize_tag_separator, normalize_decode_entities, normalize_transliterate)

            # dataset.createOrReplaceTempView(tempview)
            if self.__tableprops["DEBUG"]:
                dataset.cache()
            
            self._settings.createTempViews(dataset, tempview)
            if tempview not in self.__src_trg_file_list:
                self.__src_trg_file_list.append(tempview)

            if self.__tableprops["DEBUG"]:
                self.debug(f"Read rdbms from {sql}", dataset, tempview)

        except Exception as e:
            Writer.logger("exception in loadsql")
            Writer.logger(e)
            success = False
        return dataset if success else None

    def buildtable(self, tablename:str, sql:str, pk:str, qryType="") -> Dict[str, Any]:
        """
            Creates a tempview from the sql and creates a table from the tempview with two columns,
            PK_KEY (concatenated records from primary keys) and REC_COUNT (number of records in the group).
            Args:
                tablename (str): Name of the tempview to be created
                sql (str): SQL query to be executed to create tempview
                pk (str): Primary key columns separated by comma in case it's composed from more than one column
            Returns:
                {
                    "TABLENAME": Name of the table with primary key and count,
                    "PK": Primary key name,
                    "DF": SparkDataframe with primary key and count 
                }
        """
        print("In Build Table....")
        if not self.__check_run():
            print(TEST_NOT_INITIATED_MESSAGE)
            return None
        
        retval = {}
        try:
            Writer.logger("build source/target query...", True)
            pk = pk.strip()
            hasnopk = Atom_UDF.iif(pk == "", True, False)
            hascompositepk = ',' in pk
            pkkey = pkcols = allcols = groupedcols = ""

            df = self._settings.sprk.sql(sql)
            #df.createOrReplaceTempView(tablename)
            self._settings.createTempViews(df, tablename)
            tmp_tablename = f"tmp_{tablename}"
            df.cache()
            cols = df.columns
            df = df[sorted(cols)]
            
            if hasnopk:
                dtypes1 = (
                    "string", "boolean", "numeric", "tinyint", "smallint",
                    "bigint", "int", "void", "integer",  "double"
                )
                dtypes2 = ("varchar", "char", "text", "float", "decimal", "doubleprecision")
                retval = {"TABLENAME": tmp_tablename, "PKEY": "PK_KEY", "DF": None}
                clean_pk_level = self.__tableprops["CLEANPKLEVEL"]
                sorted_fields = sorted(df.schema.fields, key=lambda s: s.name.lower())
                for s in sorted_fields:
                    if s.name.lower() in self.__tableprops["EXCLUDECOLS-COLTEST"]:
                        continue

                    groupedcols += "`" + s.name.lower() + '`,'
                    simple = s.dataType.simpleString()
                    lower = simple.lower()
                    
                    if lower in dtypes1 or simple.startswith(dtypes2):
                        pkkey += "nvl(trim(cast(`{0}` as string)),''),'|',".format(s.name)
                        pkcols += s.name + ','
                        if pk.strip().lower() == s.name.lower():
                            allcols += "pk(nvl(trim(cast(`{0}` as string)),''), '{1}') as `{0}`,".format(
                                s.name, clean_pk_level)
                        else:
                            allcols += "cleantabledelimiters(nvl(cast(`{0}` as string),'')) as `{0}`,".format(s.name)
                    elif lower == 'date':
                        pkkey += "nvl(trim(cast(date_format(`{0}`, 'yyyy-MM-dd') as string)),''),'|',".format(s.name)
                        pkcols += s.name + ','

                        if pk.strip().lower() == s.name.lower():
                            allcols += "pk(nvl(trim(cast(date_format(`{0}`,'yyyy-MM-dd') as string)),''),'{1}') as `{0}`,".format(
                                s.name, clean_pk_level)
                        else:
                            allcols += "cleantabledelimiters(nvl(cast(date_format(`{0}`, 'yyyy-MM-dd') as string),'')) as `{0}`,".format(s.name)
                    elif s.dataType.simpleString().lower() == 'timestamp':
                        pkkey += "nvl(trim(cast(date_format(`{0}`,'yyyy-MM-dd HH:mm:ss.SSS') as string)),''),'|',".format(s.name)
                        pkcols += s.name + ','

                        if pk.strip().lower() == s.name.lower():
                            allcols += "pk(nvl(trim(cast(date_format(`{0}`,'yyyy-MM-dd " \
                                        "HH:mm:ss.SSS') as string)),''),'{1}') as `{0}`,".format(s.name, clean_pk_level)
                        else:
                            allcols += "cleantabledelimiters(nvl(cast(date_format(`{0}`," \
                                        "'yyyy-MM-dd HH:mm:ss') as string),'')) as `{0}`,".format(s.name)
                    elif s.dataType.simpleString().lower().startswith('array'):
                        pkkey += "nvl(trim(cast(concat_ws(';', sort_array(cast(`{0}` as array<string>))) as string)),''),'|',".format(s.name)
                        pkcols += s.name + ','

                        if pk.strip().lower() == s.name.lower():
                            allcols += "pk(nvl(trim(cast(concat_ws(';', sort_array(cast(`{0}` as array<string>))) as string)),'')," \
                                        "'{1}') as `{0}`,".format(s.name, clean_pk_level)
                        else:
                            allcols += "cleantabledelimiters(nvl(cast(concat_ws(';', sort_array(cast(`{0}` as array<string>))) as " \
                                        "string),'')) as `{0}`,".format(s.name)
                    else:
                        pkkey += "nvl(trim(cast(hash(`{0}`) as string)),''),'|',".format(s.name)
                        pkcols += s.name + ','
                        if pk.strip().lower() == s.name.lower():
                            allcols += "cleantabledelimiters(nvl(trim(cast(hash(pk(`{0}`, '{1}')) as string)),'')) as `{0}`,".format(
                                s.name, clean_pk_level)
                        else:
                            allcols += "cleantabledelimiters(nvl(cast(hash(`{0}`) as string),'')) as `{0}`,".format(s.name)

                col_order = pkcols[0:-1] #last char is ","
                Writer.logger('Key Column order =>' + col_order)
                del df
                gc.collect()
                #df = self._settings.sprk.sql(f"select count(1) as r_count from {tablename}")
                #df.cache()
                #v = df.first()[0]

                #del df
                #gc.collect()
                Writer.logger(
                    "Test qry => select {0} from <<src/trg_table_name>> where concat({1})='<<value>>'".format(
                        col_order, col_order.replace(",", ", '|' ,")
                    )
                )

                print(f"PK: {pkkey}")
                tmp_qry = "select pk(concat({0}),'{4}') as PK_KEY, count(1) as REC_COUNT from {2} t group by {3}".format(
                    pkkey.strip()[0:-5],
                    allcols.strip()[0:-1],
                    tablename,
                    groupedcols.strip()[0:-1],
                    self.__tableprops["CLEANPKLEVEL"]
                )
                df = self._settings.sprk.sql(tmp_qry)
                df.cache()
            elif hascompositepk:
                retval = {"TABLENAME": tmp_tablename, "PKEY": "PK_KEY", "DF": None}
                del df
                gc.collect()

                #Writer.logger("counting columns using select", True)
                #df = self._settings.sprk.sql(f"select count(1) as r_count from {tablename}")
                #df.cache()
                #v = df.first()[0]

                #del df
                #gc.collect()

                df = self._settings.sprk.sql(
                    "select pk(trim(concat({0})),'{3}') as PK_KEY, {1} from {2} t".format(
                        pk, '*', tablename, self.__tableprops["CLEANPKLEVEL"]
                    )
                )
                df.cache()
            else:
                retval = {"TABLENAME": tablename, "PKEY": pk, "DF": None}
                del df
                gc.collect()
                #df = self._settings.sprk.sql("select count(1) as count from {1} t".format('*', tablename))
                #df.cache()
                #v = df.first()[0]

                #del df
                #gc.collect()

                df = self._settings.sprk.sql("select {0} from {1} t".format('*', tablename))
                df.cache()
            
            #print(f"count1==={v}")
            #print("create temp view")
            print("Create temp view")
            df.createOrReplaceTempView(tmp_tablename)
            #self._settings.createTempViews(df, tmp_tablename)

            retval["DF"] = df
            if self.__tableprops["DEBUG"]:
                Writer.logger(f"{qryType}query ok...")
                Writer.logger(f"Records => {df.count()}")
                no_pk_msg = f"No key is given, created a composite key: {pkkey.strip()[0:-5]}"
                Writer.logger(no_pk_msg if hasnopk else f"Given key is: {pk}")
                # Writer.logger("**SQL**")
                # Writer.logger(sql)
                # Writer.logger("**end of sql**")
                Writer.logger(self.__getDFSchema(df))
                Writer.logger(self.__getDFSamples(df))
        except Exception as e:
            print(e)
            del df
            gc.collect()
            self._settings.cleartempviews()
            Writer.logger("exception in buildtable", True)
            self._settings.sprk.sql(f"drop view if exists {tablename}")
            Writer.logger(e)
            retval = {}
        
        if not retval:
            print(TEST_NOT_INITIATED_MESSAGE)
        return retval

    def comparecolumns(self, srctablename:str, trgtablename:str):
        if not self.__check_run():
            print(TEST_NOT_INITIATED_MESSAGE)
            return None
        try:
            if not (self.__resdetailsummary['COL_TEST']['START_STATUS'] and
                    self.__resdetailsummary['COL_TEST']['END_STATUS']):
                comp_cols_onfiles = (srctablename in self.__src_trg_file_list) and (trgtablename in self.__src_trg_file_list)
                if comp_cols_onfiles:
                    flag_str = "on files"
                    self.__resdetailsummary['COL_TEST']['TEST_TYPE']="F"
                else:
                    flag_str = "on query"
                    self.__resdetailsummary['COL_TEST']['TEST_TYPE'] = "Q"

                sdf = None
                tdf = None
                coltestres = None
                Writer.logger(f"starting column test... {flag_str}", True)
                if self.__tableprops["COLTEST"]:
                    self.__resdetailsummary['COL_TEST']['START_STATUS'] = True
                    self.__resdetailsummary['COL_TEST']['START_TIME'] = Atom_UDF.gettimestamp()
                    excollist = "','".join(self.__tableprops["EXCLUDECOLS-COLTEST"])
                    datatypesql = ''

                    if self.__tableprops["COLTEST-DATATYPE"]:
                        datatypesql = "and lower(st.data_type)=lower(tt.data_type)"

                    sdf = self._settings.sprk.sql(f"desc {srctablename}")
                    sdf.cache()
                    # srcCols = sdf.count()
                    self.__resdetailsummary['COL_TEST']['SRC_COL_COUNT'] = sdf.count()
                    #if comp_cols_onfiles:
                    self.__ressummary['MD_SRC_COL_COUNT'] = self.__ressummary['MD_SRC_COL_COUNT'] + \
                                                            self.__resdetailsummary['COL_TEST']['SRC_COL_COUNT']
                    # sdf.createOrReplaceTempView(f"{srctablename}_ct")
                    self._settings.createTempViews(sdf, f"{srctablename}_ct")

                    tdf = self._settings.sprk.sql(f"desc {trgtablename}")
                    tdf.cache()
                    # trgCols = tdf.count()
                    self.__resdetailsummary['COL_TEST']['TRG_COL_COUNT'] = tdf.count()
                    #if comp_cols_onfiles:
                    self.__ressummary['MD_TRG_COL_COUNT'] = self.__ressummary['MD_TRG_COL_COUNT'] + \
                                                            self.__resdetailsummary['COL_TEST']['TRG_COL_COUNT']
                    # tdf.createOrReplaceTempView(f"{trgtablename}_ct")
                    self._settings.createTempViews(tdf, f"{trgtablename}_ct")

                    if len(self.__tableprops["EXCLUDECOLS-COLTEST"]) > 0:
                        sdf = self._settings.sprk.sql(
                            f"select * from {srctablename}_ct where lower(col_name) not in ('{excollist}')")
                        # sdf.createOrReplaceTempView(f"{srctablename}_ct")
                        self.__resdetailsummary['COL_TEST']['SRC_COL_COUNT'] = sdf.count()
                        sdf.cache()
                        self._settings.createTempViews(sdf, f"{srctablename}_ct")

                        tdf = self._settings.sprk.sql(
                            f"select * from {trgtablename}_ct where lower(col_name) not in ('{excollist}')")
                        # tdf.createOrReplaceTempView(f"{trgtablename}_ct")
                        self.__resdetailsummary['COL_TEST']['TRG_COL_COUNT'] = tdf.count()
                        tdf.cache()
                        self._settings.createTempViews(tdf, f"{trgtablename}_ct")

                    del sdf
                    del tdf
                    gc.collect()

                    coltestsql = "with col_test as (select lower(st.col_name) as src_col_name, lower(st.data_type) " \
                                    "as src_data_type, lower(tt.col_name) as trg_col_name, lower(tt.data_type) as \n " \
                                    f"trg_data_type from {srctablename}_ct st full outer join {trgtablename}_ct tt " \
                                    f"on(lower(st.col_name)=lower(tt.col_name) " \
                                    f"{datatypesql})) \n select src_col_name as col_name, src_data_type as data_type, " \
                                    "'Not Found in Target' as error_desc " \
                                    "from col_test where (trg_col_name is null or trg_data_type is null)  \n " \
                                    " union all \n select trg_col_name as col_name, trg_data_type as data_type, " \
                                    "'Not Found in Source' as error_desc from col_test where " \
                                    "(src_col_name is null or src_data_type is null)\n"

                    coltestres = self._settings.sprk.sql(coltestsql)
                    coltestres.cache()
                    coltesterrors = coltestres.count()
                    # coltesterrors = coltesterrors the nbloe code is not requireed but leaving it as is
                    self.__resdetailsummary['COL_TEST']['COLTEST_ERRORS'] = \
                        self.__resdetailsummary['COL_TEST']['COLTEST_ERRORS'] + coltesterrors

                    self.__resdetailsummary['COL_TEST']['CUR_COLTEST_ERRORS'] = coltesterrors

                    self.__ressummary['COLTEST_ERRORS'] = self.__ressummary['COLTEST_ERRORS'] + coltesterrors

                    if coltesterrors > 0:
                        Writer.logger("writing logs...", True)
                        Writer.qa_metadatacomparison(coltestres, self.__tableprops)
                        Writer.logger(f"\t{coltesterrors} records added.", True)
                        Writer.logger("writing logs...completed", True)

                        if self.__tableprops["DEBUG"]:
                            Writer.logger("Columns Test")
                            Writer.logger(f"{coltesterrors} mismatches in column test")
                            # coltestres.show(2, truncate=False, vertical=True)
                            Writer.logger(self.__getDFSamples(coltestres))
                    else:
                        Writer.logger("No mismatches in column test", True)


                    Writer.logger("column test completed", True)

                    # drop the tempviews
                    self._settings.cleartempview(f"{srctablename}_ct")
                    self._settings.cleartempview(f"{trgtablename}_ct")


                    self.__resdetailsummary['COL_TEST']['END_STATUS'] = True
                    et = Atom_UDF.gettimestamp()
                    self.__resdetailsummary['COL_TEST']['END_TIME'] = et
                    self.__resdetailsummary['END_TIME'] = et
                else:
                    Writer.logger("skpping column test")

                #del coltestres
                #del sdf
                #del tdf
                #gc.collect()
                return self.__resdetailsummary['COL_TEST']
            else:
                print("Col test is already completed.")
                return self.__resdetailsummary['COL_TEST']
        except Exception as e:
            Writer.logger("exception in comparecolumns", True)
            Writer.logger(e, True)
            self._settings.cleartempview(f"{srctablename}_ct")
            self._settings.cleartempview(f"{trgtablename}_ct")

            del coltestres
            del sdf
            del tdf
            gc.collect()

            self._settings.cleartempview(f"{srctablename}_ct")
            self._settings.cleartempview(f"{trgtablename}_ct")

            self.__resdetailsummary['COL_TEST']['END_STATUS'] = False
            self.__resdetailsummary['COL_TEST']['EXCEPTION'] = str(e)
            # drop the tempviews
            self._settings.sprk.sql(f"drop view if exists {srctablename}_ct")
            self._settings.sprk.sql(f"drop view if exists {trgtablename}_ct")
            et = Atom_UDF.gettimestamp()
            self.__resdetailsummary['COL_TEST']['END_TIME'] = et
            self.__resdetailsummary['END_TIME'] = et
            return self.__resdetailsummary['COL_TEST']
    

    def comparekeycolumn(self, src:Dict[str, Any], trg:Dict[str, Any]):
        def get_errors_write_key_comparison(
                tableprops:Dict[str, Any], key_query,
                test_type:Literal["unmatching records", "duplicates in source", "duplicates in target"]
            ) -> int:
            self._settings.setsparkobject(SparkSession.builder.getOrCreate())
            print(f"Test type: {test_type}\nCreating keyres")
            keyres = self._settings.sprk.sql(key_query)
            keyres.cache()

            print("Counting key errors")
            keyerrors = keyres.count()
            if keyerrors:
                Writer.logger("writing logs...", True)
                Writer.qa_keycomparison(keyres, tableprops)
                Writer.logger(f"\t{keyerrors} records added.", True)
                Writer.logger("writing logs...completed", True)

                get_iif = lambda dct: Atom_UDF.iif(dct["PKEY"] == '', 'PK_KEY', dct["PKEY"])
                if tableprops["DEBUG"]:
                    Writer.logger(self.__getDFSamples(keyres))
                    Writer.logger("Source: {0}.{1}".format(src["TABLENAME"], get_iif(src)), True)
                    Writer.logger("Target: {0}.{1}".format(trg["TABLENAME"], get_iif(trg)), True)
            else:
                Writer.logger(f"No mismatches in key test for {test_type}", True)
            return keyerrors
        
        def __set_details_summary(d_test_st, d_test_ss:bool, d_test_es:bool, dct:Dict[str, Any], msg:str = None):
            dct['KEY_TEST']['END_STATUS'] = msg is not None
            dct['DATA_TEST']['START_TIME'] = d_test_st
            dct['DATA_TEST']['START_STATUS'] = d_test_ss
            dct['DATA_TEST']['END_STATUS'] = d_test_es

            ts = Atom_UDF.gettimestamp()
            dct['KEY_TEST']['END_TIME'] = ts
            dct['DATA_TEST']['END_TIME'] = ts

            if msg:
                dct['KEY_TEST']['EXCEPTION'] = msg
            return dct
        
        if not self.__check_run():
            print(TEST_NOT_INITIATED_MESSAGE)
            return None
        try:
            if self.__resdetailsummary['KEY_TEST']['START_STATUS'] and self.__resdetailsummary['KEY_TEST']['END_STATUS']:
                print("key test is already completed.")
            
            Writer.logger("testing key column...", True)
            self.__resdetailsummary['KEY_TEST']['START_STATUS'] = True
            keytest_start_time = Atom_UDF.gettimestamp()
            self.__resdetailsummary['KEY_TEST']['START_TIME'] = keytest_start_time
            Writer.logger("checking count src vs target...", True)

            src["DF"].cache()
            trg["DF"].cache()

            sdc = src["DF"].count()
            Writer.logger(f"source count {sdc}")
            tdc = trg["DF"].count()
            Writer.logger(f"target count {tdc}")

            if sdc > 0 and tdc > 0:
                # if data set has records
                Writer.logger("starting key test...", True)

                kt_notfound = "with a as (select distinct cast(nvl(st.{0}, '') as string) as source_table_key, " \
                                "cast(nvl(tt.{2},'') as string) as target_table_key " \
                                "from {1} st full join {3} tt on st.{0}=tt.{2} " \
                                "where st.{0} is null or tt.{2} is null), " \
                                "srcvstrg as (select (case when source_table_key='' then target_table_key " \
                                "when target_table_key='' then source_table_key else '' end) as keycol, " \
                                "(case  when source_table_key='' then 'NOT FOUND IN SOURCE TABLE' " \
                                "when target_table_key='' then 'NOT FOUND IN TARGET TABLE' else '' end) as errortype " \
                                "from a where a.source_table_key!=a.target_table_key) select * from srcvstrg" \
                    .format(src["PKEY"], src["TABLENAME"], trg["PKEY"], trg["TABLENAME"])

                kt_dupsinsrc = "with dupsinsrc as (select cast(st.{0} as string) as keycol, " \
                                "'DUPLICATES IN SOURCE' as errortype " \
                                "from {1} st where st.{0} is not null " \
                                "group by st.{0} having count(st.{0})!=1) select * from dupsinsrc" \
                    .format(src["PKEY"], src["TABLENAME"])

                kt_dupsintrg = "with dupsintarg as (select cast(tt.{0} as string) as keycol, " \
                                "'DUPLICATES IN TARGET' as errortype " \
                                "from {1} tt where tt.{0} is not null " \
                                "group by tt.{0} having count(tt.{0})!=1) select * from dupsintarg" \
                    .format(trg["PKEY"], trg["TABLENAME"])

                keymatchessql = f"select count(1) as match_count from {src["TABLENAME"]} st, {trg["TABLENAME"]} tt "
                keymatchessql += f"where st.{src["PKEY"]}=tt.{trg["PKEY"]}"

                try:
                    Writer.logger("checking for matches...", True)
                    keyres = self._settings.sprk.sql(keymatchessql)

                    print("Keyres",keyres.first()[0])
                    keyres.cache()
                    keymatches = keyres.first()[0]
                    self.__resdetailsummary['KEY_TEST']['MATCH_FOUND'] = keymatches
                    del keyres
                    gc.collect()
                    Writer.logger(f"Total Matches= {keymatches}", True)

                    try:
                        Writer.logger("testing key column for unmatching records in source vs target...", True)
                        keyerrortotal = get_errors_write_key_comparison(self.__tableprops, kt_notfound, "unmatching records")

                        Writer.logger("testing key column for duplicates in source...", True)
                        keyerrortotal += get_errors_write_key_comparison(self.__tableprops, kt_dupsinsrc, "duplicates in source")

                        Writer.logger("testing key column for duplicates in target...")
                        keyerrortotal += get_errors_write_key_comparison(self.__tableprops, kt_dupsintrg, "duplicates in target")

                        Writer.logger(f"total key errors {keyerrortotal}", True)

                    except Exception as e:
                        if "DF" in src.keys():
                            del src["DF"]
                        if "DF" in trg.keys():
                            del trg["DF"]
                        del keyres
                        gc.collect()
                        Writer.logger("exception in comparekeycolumn (1)", True)
                        Writer.logger(e)
                        keyerrors = 0

                    self.__resdetailsummary['KEY_TEST']['KEYTEST_ERRORS'] += keyerrortotal
                    self.__ressummary['KEYTEST_ERRORS'] += keyerrortotal
                    Writer.logger("key test completed successfully", True)
                    self.__resdetailsummary['KEY_TEST']['END_STATUS'] = True
                    self.__resdetailsummary['KEY_TEST']['END_TIME'] = Atom_UDF.gettimestamp()
                    self.__resdetailsummary['END_TIME'] = Atom_UDF.gettimestamp()

                except Exception as e:
                    keyerrors = 0
                    keyerrortotal = 0
                    if "DF" in src.keys():
                        del src["DF"]
                    if "DF" in trg.keys():
                        del trg["DF"]
                    del keyres
                    gc.collect()
                    Writer.logger("exception in comparekeycolumn (2)", True)
                    Writer.logger(e)
                    self.__resdetailsummary['KEY_TEST']['MATCH_FOUND'] = 0
                    self.__resdetailsummary['KEY_TEST'] = False
                    self.__resdetailsummary['KEY_TEST']['EXCEPTION'] = str(e)
                    
                    et = Atom_UDF.gettimestamp()
                    self.__resdetailsummary['KEY_TEST']['END_TIME'] = et
                    self.__resdetailsummary['END_TIME'] = et

            elif sdc==0 and tdc==0:
                if self.__tableprops["COLTEST"]:
                    tp_col_test = self.__resdetailsummary['COL_TEST']
                    # Col test is set to true and status is pass
                    if (tp_col_test['END_STATUS'] and tp_col_test['CUR_COLTEST_ERRORS']==0):

                        Writer.logger(COL_TEST_LOG.format(result = "passed"), True)
                        self.__resdetailsummary['DATA_TEST']['SRC_COL_COUNT'] = tp_col_test['SRC_COL_COUNT']
                        self.__resdetailsummary['DATA_TEST']['TRG_COL_COUNT']= tp_col_test['TRG_COL_COUNT']
                        self.__resdetailsummary = __set_details_summary(keytest_start_time, True, True, self.__resdetailsummary)
                    else:
                        # Col test is set to true and status is fail
                        Writer.logger(COL_TEST_LOG.format(result = "failed"), True)
                        self.__resdetailsummary = __set_details_summary(keytest_start_time, False, False, self.__resdetailsummary, COL_TEST_LOG.format(result = "failed (cannot verify)"))
                else:
                    # Col test is set to false
                    Writer.logger(COL_TEST_LOG.format(result = "not done"), True)
                    self.__resdetailsummary = __set_details_summary(keytest_start_time, False, False, self.__resdetailsummary, COL_TEST_LOG.format(result = "not done"))
                self.__resdetailsummary['END_TIME'] = Atom_UDF.gettimestamp()
            else:
                if "DF" in src.keys():
                    del src["DF"]
                if "DF" in trg.keys():
                    del trg["DF"]
                gc.collect()
                Writer.logger(f"cannot perform key test", True)
                Writer.logger(f"{sdc} records found in source, {tdc} records in target.", True)

                self.__resdetailsummary['KEY_TEST']['END_STATUS'] = True
                self.__resdetailsummary['KEY_TEST']['END_TIME'] = self.__resdetailsummary['KEY_TEST']['START_TIME']
                self.__resdetailsummary['END_TIME'] = Atom_UDF.gettimestamp()

        except Exception as e:
            if "DF" in src.keys():
                del src["DF"]
            if "DF" in trg.keys():
                del trg["DF"]
            gc.collect()
            Writer.logger("exception in comparekeycolumn", True)
            Writer.logger(e)
            self.__resdetailsummary['END_TIME'] = Atom_UDF.gettimestamp()
        return self.__resdetailsummary['KEY_TEST']

    def comparedata(self, src:Dict[str, Any], trg:Dict[str, Any]):
        if not self.__check_run():
            print(TEST_NOT_INITIATED_MESSAGE)
            return None
        
        data_test = self.__resdetailsummary['DATA_TEST']
        try:
            if data_test['START_STATUS'] or data_test['END_STATUS']:
                print("data test is already completed.")
                return self.__resdetailsummary['DATA_TEST']
            
            Writer.logger("comparing data...", True)
            data_test['START_STATUS'] = True
            data_test['START_TIME'] = Atom_UDF.gettimestamp()
            data_test['FAILED_COLS'] = 0

            sqcode = tqcode = ''
            src["DF"].cache()
            sdc = 0 if src is None else src["DF"].count()

            trg["DF"].cache()
            tdc = 0 if trg is None else trg["DF"].count()

            Writer.logger(f"source count {sdc}", True)
            Writer.logger(f"target count {tdc}", True)

            stablename = Atom_UDF.iif(src["TABLENAME"] == '', 'defsrc', src["TABLENAME"])
            ttablename = Atom_UDF.iif(trg["TABLENAME"] == '', 'deftrg', trg["TABLENAME"])

            exclude_cols = self.__tableprops["EXCLUDECOLS-DATATEST"]
            get_count = lambda d: len(d["DF"].columns) - len(Atom_UDF.getCommonValues(d["DF"].columns, exclude_cols))
            cols = Atom_UDF.getCommonValues(src["DF"].columns, trg["DF"].columns)
            if len(cols)>1:
                data_test['SRC_COL_COUNT'] = get_count(src)
                data_test['TRG_COL_COUNT'] = get_count(trg)
                if sdc > 0 and tdc > 0:
                    Writer.logger('starting data test...', True)
                    # debug = self.__tableprops["DEBUG"]
                    spk = str(src["PKEY"]).lower()
                    tpk = str(trg["PKEY"]).lower()
                    compop = Atom_UDF.iif(self.__tableprops["REVERSECHK"], "", "!")

                    if len(cols) > 0:
                        data_test['SRC_REC_COUNT'] = sdc
                        data_test['TRG_REC_COUNT'] = tdc
                        self.__ressummary['DT_SRC_COL_COUNT'] += data_test['SRC_COL_COUNT']
                        self.__ressummary['SRC_REC_COUNT'] += data_test['SRC_REC_COUNT']
                        self.__ressummary['DT_TRG_COL_COUNT'] += data_test['TRG_COL_COUNT']
                        self.__ressummary['TRG_REC_COUNT'] += data_test['TRG_REC_COUNT']

                        datatesttql = datatesttql_all = ""
                        src_df_schema = Atom_UDF.getdfschema(src["DF"])
                        trg_df_schema = Atom_UDF.getdfschema(trg["DF"])

                        def get_qcode_date(
                                column:str,
                                dtype:Literal["date", "timestamp","void"],
                                table:Literal["src", "trg"]
                            ) -> str:
                            col_name = "source_data" if table=="src" else "target_data"
                            alias   = "st" if table=="src" else "tt"
                            dtformat = "yyyy-MM-dd" if dtype == "date" else "yyyy-MM-dd HH:mm:ss.SSS"

                            if dtype in ("date", "timestamp"):
                                return f"nvl(cast(date_format({alias}.`{column}`, '{dtformat}') as string),'') as {col_name}"
                            if dtype == "void":
                                return f"nvl(cast({alias}.`{column}` as string),'') as {col_name}"
                            qcode = f"(case when isdate({alias}.`{column}`) then nvl(cast(date_format({alias}.`{column}`,"
                            qcode += f"'yyyy-MM-dd HH:mm:ss.SSS') as string),'') else nvl(cast({alias}.`{column}` as string), '') "
                            return qcode + f"end ) as {col_name}"
                        
                        def get_qcode_arr(column:str, dtype:str, table:Literal["sre", "trg"]):
                            col_name = "source_data" if table=="src" else "target_data"
                            alias    = f"st.`{column}`" if table=="src" else f"tt.`{column}`"
                            if dtype.startswith("array"):
                                return f"nvl(cast(concat_ws(';', sort_array(cast({alias} as array<string>))) as string),'') as {col_name}"
                            if dtype == "void":
                                return f"nvl(cast({alias} as string),'') as {col_name}"
                            return f"nvl(cast(concat_ws(';', sort_array(split({alias}, ';'))) as string),'') as {col_name}"
                        
                        def get_data_query(column:str, sqcode:str, tqcode:str, src_pk:str, trg_pk:str, str_tbl:str, trg_tbl:str):
                            inner = f"select '{column}' as field_name, st.{spk} as keycol,  {sqcode}, {tqcode} "
                            inner += f"from {stablename} st, {ttablename} tt "
                            inner += f"where nvl(cast(st.{spk} as string),'') = nvl(cast(tt.{tpk} as string),'')"
                            return f"select * from ({inner}) `tbl_{column}` where source_data {compop}= target_data\n union all \n"
                        
                        for column in cols:
                            if column not in (spk, tpk) and column not in exclude_cols:
                                #sdatatypestring = src["DF"].schema[column].dataType.simpleString().lower()
                                #tdatatypestring = trg["DF"].schema[column].dataType.simpleString().lower()

                                sdatatypestring = src_df_schema[column]["col_datatype"]
                                tdatatypestring = trg_df_schema[column]["col_datatype"]
                                
                                dtypes = ('string', 'boolean', 'numeric', 'tinyint', 'smallint', 'bigint', 'int', 'integer', 'double', 'void')
                                date_types = ('date', 'timestamp', 'void')
                                starts_with_dtypes = ('varchar', 'char', 'text', 'float', 'decimal', 'doubleprecision')
                                
                                long_cond = lambda s: (s in dtypes or s.startswith(starts_with_dtypes))
                                arr_cond = lambda s: s.startswith('array') or s == "void"
                                if long_cond(sdatatypestring) and long_cond(tdatatypestring):
                                    sqcode = f"nvl(cast(st.`{column}` as string),'') as source_data"
                                    tqcode = f"nvl(cast(tt.`{column}` as string),'') as target_data"
                                elif sdatatypestring in date_types or tdatatypestring in date_types:
                                    sqcode = get_qcode_date(column, sdatatypestring, "src")
                                    tqcode = get_qcode_date(column, tdatatypestring, "trg")
                                elif arr_cond(sdatatypestring) or arr_cond(tdatatypestring):
                                    sqcode = get_qcode_arr(column, sdatatypestring, "src")
                                    tqcode = get_qcode_arr(column, tdatatypestring, "trg")
                                else:
                                    sqcode = f"nvl(cast(hash(st.`{column}`) as string),'') as source_data"
                                    tqcode = f"nvl(cast(hash(tt.`{column}`) as string),'') as target_data"
                                datatesttql = get_data_query(column, sqcode, tqcode, spk, tpk, stablename, ttablename)

                                if (self._settings.runallcols):
                                    datatesttql_all += datatesttql
                                else:
                                    Writer.logger(f"...checking column '{column}'", True)
                                    dtest = self._settings.sprk.sql(datatesttql[0:-12].strip())
                                    dtest.cache()
                                    datamismatches = dtest.count()

                                    data_test['DATATEST_ERRORS'] += datamismatches
                                    self.__ressummary['DATATEST_ERRORS'] += datamismatches

                                    if datamismatches > 0:
                                        Writer.logger(f"writing logs for '{column}'...", True)
                                        Writer.logger(f"\t {datamismatches} mismatches found in '{column}'.", True)
                                        data_test['FAILED_COLS'] +=1

                                        Writer.qa_datacomparison(dtest, self.__tableprops)
                                        if self.__tableprops["DEBUG"]:
                                            Writer.logger(self.__getDFSamples(dtest))
                                    else:
                                        Writer.logger(f"\t NO mismatches found in '{column}'.", True)

                                    #del dtest
                                    #gc.collect()

                        if (self._settings.runallcols):
                            Writer.logger(f"...checking table {self.__tableprops['TABLENAME']}", True)
                            dtest = self._settings.sprk.sql(datatesttql_all[0:-12].strip())
                            dtest.cache()
                            datamismatches = dtest.count()
                            data_test['DATATEST_ERRORS'] = data_test['DATATEST_ERRORS'] + datamismatches

                            self.__ressummary['DATATEST_ERRORS'] = self.__ressummary['DATATEST_ERRORS'] + datamismatches
                            if datamismatches > 0:
                                Writer.logger(f"writing logs for '{self.__tableprops['TABLENAME']}'...", True)
                                Writer.logger(f"\t {datamismatches} mismatches found in '{self.__tableprops['TABLENAME']}'.", True)

                                dtest.createOrReplaceTempView("f_cols")
                                #dtest.show()
                                failedCols_count= self._settings.sprk.sql("select count(distinct field_name) from f_cols").first()[0]
                                data_test['FAILED_COLS'] =failedCols_count
                                self._settings.sprk.sql(f"drop view if exists f_cols")

                                Writer.qa_datacomparison(dtest, self.__tableprops)
                                if self.__tableprops["DEBUG"]:
                                    # dtest.show(2, truncate=False, vertical=True)
                                    Writer.logger(self.__getDFSamples(dtest))
                            else:
                                Writer.logger(f"\t NO mismatches found in '{self.__tableprops['TABLENAME']}'.", True)

                            #del dtest
                            #gc.collect()

                        Writer.logger("data test completed.", True)
                        if (data_test['SRC_COL_COUNT'] != data_test['TRG_COL_COUNT']):
                            Writer.logger("pair query has column mismatch, orphan columns ignored.", True)

                        data_test['END_STATUS'] = True
                        et = Atom_UDF.gettimestamp()
                        data_test['END_TIME'] = et
                        self.__resdetailsummary['END_TIME'] = et
                    else:
                        Writer.logger("no common cols in source & target.", True)
                        data_test['END_STATUS'] = False
                        data_test['EXCEPTION'] = 'no common cols in source & target.'
                        et = Atom_UDF.gettimestamp()
                        data_test['END_TIME'] = et
                        self.__resdetailsummary['END_TIME'] = et
                elif (sdc == 0 and tdc==0):
                    if self.__tableprops["COLTEST"]:
                        if (self.__resdetailsummary['COL_TEST']['END_STATUS'] and
                                self.__resdetailsummary['COL_TEST']['CUR_COLTEST_ERRORS']==0):
                            Writer.logger(COL_TEST_LOG.format(result = "passed"), True)
                            data_test['END_STATUS'] = True
                            data_test['END_TIME'] = Atom_UDF.gettimestamp()
                        else:
                            Writer.logger(COL_TEST_LOG.format(result = "failed"), True)
                            data_test['END_STATUS'] = False
                            data_test['END_TIME'] = Atom_UDF.gettimestamp()
                            data_test['EXCEPTION']= COL_TEST_LOG.format(result = "failed (cannot verify)")
                    else:
                        Writer.logger(COL_TEST_LOG.format(result = "failed"), True)
                        data_test['END_STATUS'] = False
                        data_test['END_TIME'] = Atom_UDF.gettimestamp()
                        data_test['EXCEPTION'] = COL_TEST_LOG.format(result = "failed (cannot verify)")
                    self.__resdetailsummary['END_TIME'] = Atom_UDF.gettimestamp()
                else:
                    if "DF" in src.keys():
                        del src["DF"]
                    if "DF" in trg.keys():
                        del trg["DF"]
                    gc.collect()
                    Writer.logger("cannot perform data comparision", True)
                    Writer.logger(f"{sdc} records found in source, {tdc} records in target.", True)
                    data_test['SRC_REC_COUNT'] = sdc  # sdf.count()
                    data_test['TRG_REC_COUNT'] = tdc  # tdf.count()
                    data_test['END_STATUS'] = False
                    data_test['END_TIME'] = data_test['START_TIME']
                    self.__resdetailsummary['END_TIME'] = Atom_UDF.gettimestamp()
            else:
                Writer.logger("cannot perform data comparision as no common cols found.", True)
                Writer.logger("src cols==>", src["DF"].columns)
                Writer.logger("trg cols==>", trg["DF"].columns)
                Writer.logger("common cols==>", cols)
                data_test['END_STATUS'] = False
                data_test['EXCEPTION'] = "cannot perform data comparision as no common cols found."
                et = Atom_UDF.gettimestamp()
                data_test['END_TIME'] = et
                self.__resdetailsummary['END_TIME'] = et

        except Exception as e:
            Writer.logger("exception in comparedata", True)
            print(e)
            Writer.logger(e)
            del dtest
            if "DF" in src.keys():
                del src["DF"]
            if "DF" in trg.keys():
                del trg["DF"]
            gc.collect()
            data_test['END_STATUS'] = False
            data_test['EXCEPTION'] = str(e)
            et = Atom_UDF.gettimestamp()
            data_test['END_TIME'] = et
            self.__resdetailsummary['END_TIME'] = et
        self.__resdetailsummary['DATA_TEST'] = data_test
        return data_test 

    def dataquality(self, dfortempview:DataFrame=None, pkey:str="", columns:str="", dqname:str="", condition:str=""):
        if not self.__check_run():
            print(TEST_NOT_INITIATED_MESSAGE)
            return None
        
        pkey = f"concat({pkey})" if "," in pkey else pkey

        columns = f"concat({columns})" if "," in columns else columns

        Writer.logger(f"data quality checks... {dqname}", True)
        resdataquality = {
            'COLNAME': '', 'COLNAME2': '', 'TABLENAME': '', 'TYPE': '',
            'CONDITION': '', 'START_STATUS': False, 'START_TIME': '', 'END_TIME': '',
            'DATAQUALITY_ERRORS': 0, 'LOG_PATH': '', 'END_STATUS': False, 'EXCEPTION': ''
        }

        if self.__ressummary['DATAQUALITY_ERRORS'] == -1:
            self.__ressummary['DATAQUALITY_ERRORS'] = 0

        if self.__ressummary['DATAQUALITY_TESTS'] == -1:
            self.__ressummary['DATAQUALITY_TESTS'] = 0
            self.__ressummary['DATAQUALITY_TEST_PASSED'] = 0
            self.__ressummary['DATAQUALITY_TEST_FAILED'] = 0

        self.__ressummary['DATAQUALITY_TESTS'] += 1

        try:
            self.__dqId += 1
            resdataquality['COLNAME'] = dqname
            resdataquality['CONDITION'] = condition
            resdataquality['START_STATUS'] = True
            resdataquality['START_TIME'] = Atom_UDF.gettimestamp()

            tempView = ""
            if isinstance(dfortempview, DataFrame):
                tempView = f"{dqname}_df"
                try:
                    # dfortempview.createOrReplaceTempView(tempView)
                    self._settings.createTempViews(dfortempview, tempView)
                except Exception as e:
                    # print("bad column name.")
                    tempView = "_df"
                    # dfortempview.createOrReplaceTempView(tempView)
                    self._settings.createTempViews(dfortempview, tempView)
            else:
                tempView = str(dfortempview)

            resdataquality['TABLENAME'] = tempView

            if self.__tableprops["DEBUG"]:
                Writer.logger("print parameters")
                Writer.logger(f"PK_KEY='{pkey}'")
                Writer.logger(f"Columns='{columns}'")
                Writer.logger(f"Name='{dqname}'")
                Writer.logger(f"Condition='{condition}'")


            if len(columns.strip()) > 0 and len(pkey.strip())>0 and len(condition.strip())>0 and len(dqname)>0:
                tmp=condition.replace("'", "''")
                try:
                    dqTest = self._settings.sprk.sql(f"select {pkey} as pkey, '{columns}' as failed_col, '{tmp}' as condition, "
                                            f"{columns} as failed_colvalue from {tempView} where not ({condition})")
                    dqTest.cache()
                    dqErrorCount = dqTest.count()
                    resdataquality['DATAQUALITY_ERRORS'] = dqErrorCount
                    self.__ressummary['DATAQUALITY_ERRORS'] += dqErrorCount
                    if dqErrorCount > 0:
                        tbl_name = self.__tableprops["TABLENAME"]
                        dq_name = dqname.replace(",", "_")

                        if dqname in self.__dqcols:
                            dq_name += f"_{self.__dqId}"
                            resdataquality['COLNAME2'] = dq_name
                            resdataquality['LOG_PATH'] = f"{tbl_name}/{dq_name}"
                        else:
                            self.__dqcols.append(dqname)
                            resdataquality['COLNAME2'] = dq_name
                            resdataquality['LOG_PATH'] = f"{tbl_name}/{dq_name}"

                        Writer.logger(f"{dqErrorCount} errors found for '{condition}' on '{dqname}'", True)

                        if self.__tableprops["DEBUG"]:
                            Writer.logger(self.__getDFSamples(dqTest))
                        Writer.qa_dataquality(dqTest, resdataquality['LOG_PATH'])
                        self.__ressummary['DATAQUALITY_TEST_FAILED'] += 1
                    else:
                        self.__ressummary['DATAQUALITY_TEST_PASSED'] += 1
                        Writer.logger(f"no errors found for {condition} on {dqname}", True)

                    resdataquality['END_STATUS'] = True
                    resdataquality['END_TIME'] = Atom_UDF.gettimestamp()

                except Exception as e:
                    self.__ressummary['DATAQUALITY_TEST_FAILED'] += 1
                    resdataquality['END_STATUS'] = False
                    resdataquality['EXCEPTION'] = str(e)
                    et = Atom_UDF.gettimestamp()
                    resdataquality['END_TIME'] = et
                    self.__resdetailsummary['END_TIME'] = et
                    Writer.logger("exception in dataquality sql..", True)
                    Writer.logger(e)
                    Writer.logger("faulty sql")
                    Writer.logger(f"select {pkey}, '{columns}' as failed_col, '{tmp}' as condition, "
                                            f"{columns} as failed_colvalue from {tempView} where not ({condition})")

                self.__resdetailsummary['DATAQUALITY'][str(self.__dqId)] = resdataquality
                et = Atom_UDF.gettimestamp()
                self.__resdetailsummary['END_TIME'] = et
                gc.collect()
                return resdataquality
            else:
                Writer.logger("pkey/colname/condition/dqname cannot be empty.", True)
                self.__resdetailsummary['DATAQUALITY'][str(self.__dqId)] = None
                et = Atom_UDF.gettimestamp()
                self.__resdetailsummary['END_TIME'] = et
                gc.collect()
                return None

        except Exception as e:
            print(str(e))
            del dqTest
            gc.collect()
            return None
        

    def finishtable(self):
        """Write the table results to qa_testruns and return the table detail summary."""
        Writer.logger("completing table...", True)
        self.logTestRuns()
        return self.__resdetailsummary

    # close test
    def finishtest(self):
        """Set the run end time and write qa_testinfosummary."""
        Writer.logger("completing test...", True)
        self.__ressummary['END_TIME'] = Atom_UDF.gettimestamp()
        Writer.qa_testinfosummary(self.__ressummary)
        return None

    # get log on exception
    def copylog(self):
        try:
            log_path = {self._settings.output_root}/{self._settings.applicationid}
            spark.read.format(self._settings.writeformat) \
                .load(f"{GFDOQA_TEST_PATH}/{self._settings.applicationid}/qalog").select("text").orderBy("id") \
                .coalesce(1).write.text(f"{log_path}/qalog")
            print(f"logs are copied to {log_path}/qalog.")
        except Exception as e:
            print(e)
        return

    def __get_sql_report(self, qa_folder:str, where_cond:str, group_cols:str = ""):
        FROM_CLAUSE = f"FROM {qa_folder} {where_cond}"
        if not group_cols: #sample_sql
            return f"SELECT * {FROM_CLAUSE}"
        return f"SELECT {group_cols}, count(1) AS error_count {FROM_CLAUSE} GROUP BY {group_cols}"

    # print sample evidence
    def printsampletestreport(self, limit:int = 5):
        self._settings.sprk = SparkSession.builder.getOrCreate()
        settings = self._settings
        # Define helpers
        def log_df_samples(df:DataFrame, title:str, limit:int):
            Writer.logger(title)
            Writer.logger("==========================")
            Writer.logger(f"Total errors logged :{df.count()}")
            Writer.logger(f"first {limit} are shown below...")
            Writer.logger(f"{self.__getDFSamples(df, limit)}\n")
            return
        
        def process_test(enabled:bool, err_count:int, qa_folder:str, summary_sql:str, sample_sql:str, title_test:str):
            if not enabled:
                Writer.logger(f"No {title_test} done.\n")
                return
            if err_count <= 0:
                Writer.logger(f"No errors found in {title_test} test.\n")
                return
            
            reader = settings.sprk.read.format(settings.writeformat)
            try:
                settings.createTempViews(reader.load(f"{settings.errorlog_base}/{qa_folder}"), qa_folder)
                log_df_samples(settings.sprk.sql(summary_sql), f"{title_test} Summary", limit)
                log_df_samples(settings.sprk.sql(sample_sql), f"{title_test} Samples", limit)
            except Exception as e:
                Writer.logger(f"{e}\n")
            return
        
        Writer.logger("printing sample evidence...", True)
        fsr = self.getsummary()
        Writer.logger("\n\nTest Evidence (samples)")
        Writer.logger("=======================\n")

        # for cols
        where_cond = f"where runid='{settings.applicationid}'"
        # Column test
        process_test(
            self.__tableprops["COLTEST"], int(fsr['COLTEST_ERRORS']), "qa_metadatacomparison",
            summary_sql = self.__get_sql_report("qa_metadatacomparison", where_cond, "tablename"),
            sample_sql = self.__get_sql_report("qa_metadatacomparison", where_cond),
            title_test = "Metadatacomparison"
        )

        # Key test
        process_test(
            self.__tableprops["DATATEST"], int(fsr['KEYTEST_ERRORS']), "qa_keycomparison",
            summary_sql = self.__get_sql_report("qa_keycomparison", where_cond, "tablename, error_msg"),
            sample_sql = self.__get_sql_report("qa_keycomparison", where_cond),
            title_test = "Keycomparison"
        )
        
        # Data test
        process_test(
            self.__tableprops["DATATEST"], int(fsr['DATATEST_ERRORS']), "qa_datacomparison",
            summary_sql = self.__get_sql_report("qa_datacomparison", where_cond, "tablename, failedcol"),
            sample_sql = self.__get_sql_report("qa_datacomparison", where_cond),
            title_test = "Datacomparison"
        )

        # dataquality test
        if self.__ressummary.get("DATAQUALITY_TESTS", 0) <= 0:
            Writer.logger(f"No dataquality test done.\n")
        elif fsr.get('DATAQUALITY_ERRORS', 0) <= 0:
            Writer.logger(f"No errors found in dataquality test.\n")
        else:
            try:
                Writer.logger("Dataquality Summary")
                Writer.logger("=====================")
                dqtests:dict = self.__resdetailsummary["DATAQUALITY"]
                print("\n\nDQ_Tests: ")
                for k,v in dqtests.items():
                    print(f"{k}:", v)
                print("\n\n")
                for dqtest in dqtests.values():
                    if dqtest is None:
                        continue
                    
                    Writer.logger(f"Columnname: '{dqtest['COLNAME']}'")
                    Writer.logger(f"Condition: '{dqtest['CONDITION']}'")
                    if not dqtest['END_STATUS']:
                        Writer.logger(f"Error: {dqtest['EXCEPTION']}\n")
                    elif dqtest['DATAQUALITY_ERRORS'] > 0:
                        Writer.logger(f"Total errors: {dqtest['DATAQUALITY_ERRORS']}, fail\n")
                    else:
                        Writer.logger("No errors found, pass.\n")
            except Exception as e:
                Writer.logger(e)

        Writer.logger("end of sample evidence\n\n")

        # print sample code
        Writer.logger("# python code to get test errors & summary")
        # testruns, testinfo and testinfosummary
        Writer.logger("%python\n\n"
                "#ATOM v{3}\n#optional\n"
                "#from pyspark.sql import SparkSession\n"
                "#spark = SparkSession.builder.getOrCreate()\n\n"
                "run_id='{2}'\n"
                "#qa_testinfo\n"
                "qa_testinfo_df=spark.read.format('{1}').load(f'{0}/{{run_id}}/summary_log/qa_testinfo')\n"
                "qa_testinfo_df.createOrReplaceTempView('qa_testinfo')\n\n"
                "#qa_testruns\n"
                "qa_testruns_df=spark.read.format('{1}').load(f'{0}/{{run_id}}/summary_log/qa_testruns')\n"
                "qa_testruns_df.createOrReplaceTempView('qa_testruns')\n\n"
                "#qa_testinfosummary\n"
                "qa_testinfosummary_df=spark.read.format('{1}').load(f'{0}/{{run_id}}/summary_log/qa_testinfosummary')\n"
                "qa_testinfosummary_df.createOrReplaceTempView('qa_testinfosummary')\n"
                .format(settings.output_root, settings.writeformat, settings.applicationid, settings.CODE_VER)
        )

        errlog_sql = []
        def log_error_block(name:str, errlog_sql:list):
            Writer.logger(
                f"# {name}\n{name}_df=spark.read.format('{settings.writeformat}')"
                f".load(f'{settings.output_root}/{{run_id}}/error_log/{name}')\n"
                f"{name}_df.createOrReplaceTempView('{name}')\n"
            )
            errlog_sql.append(f"select * from {name} {where_cond}';")
            return errlog_sql
        
        # metadata
        if self.__resdetailsummary['COL_TEST']['COLTEST_ERRORS']>0:
            errlog_sql = log_error_block("qa_metadatacomparison", errlog_sql)

        # keytest
        if self.__resdetailsummary['KEY_TEST']['KEYTEST_ERRORS']>0:
            errlog_sql = log_error_block("qa_keycomparison", errlog_sql)

        # datatest
        if self.__resdetailsummary['DATA_TEST']['DATATEST_ERRORS']>0:
            errlog_sql = log_error_block("qa_datacomparison", errlog_sql)

        # dataquality test
        dqSQL = []
        if self.__ressummary.get('DATAQUALITY_ERRORS', 0) > 0:
            Writer.logger("#qa_dataquality")
            for dqtest in self.__resdetailsummary["DATAQUALITY"].values():
                if not dqtest or dqtest['DATAQUALITY_ERRORS'] <= 0:
                    continue
                if not dqtest['END_STATUS']:
                    Writer.logger(dqtest['EXCEPTION'])
                    continue
                
                Writer.logger(f"'#{dqtest['COLNAME']}'")
                Writer.logger(
                    "qa_dataquality_{2}=spark.read.format('{1}').load(f'{0}/{{run_id}}/error_log/qa_dataquality/{3}')\n"
                    "qa_dataquality_{2}.createOrReplaceTempView('qa_dataquality_{2}')\n"
                    .format(settings.output_root, settings.writeformat, dqtest['COLNAME2'],dqtest['LOG_PATH'])
                )
                dqSQL.append(f"select * from qa_dataquality_{dqtest['COLNAME2']};")
        
        try:
            Writer.logger("\n###################################################################################################")
            Writer.logger("%sql\n--summary")
            Writer.logger(f"select * from qa_testinfo {where_cond};")
            Writer.logger(f"select * from qa_testinfosummary {where_cond};")
            Writer.logger(f"select * from qa_testruns {where_cond};\n")

            get_msg = lambda test: f"--no errors reported in {test} comparison  logs\n"
            Writer.logger("--error logs\n{0}".format("\n".join(errlog_sql)) if errlog_sql else get_msg("key/data"))
            Writer.logger("--data quality logs\n{0}".format("\n".join(dqSQL)) if dqSQL else get_msg("data quality"))
            
            log_path = f"{settings.output_root}/{settings.applicationid}"
            settings.sprk.read.format(settings.writeformat) \
                .load(f"{GFDOQA_TEST_PATH}/{settings.applicationid}/qalog").select("text").orderBy("id") \
                .coalesce(1).write.text(f"{log_path}/qalog")

            #dbutils.fs.cp(settings.logfile, f"{log_path}")
            dbutils.fs.cp(f"{GFDOQA_TEST_PATH}/{settings.applicationid}", f"{log_path}", True)
            dbutils.fs.rm(f"{GFDOQA_TEST_PATH}/{settings.applicationid}", True)
            
            settings.logfile = ""
            self._settings = settings

            Writer.logger("test completed...", True)

            self.generate_html(limit)
        except Exception as e:
            print(e)
        return

    def generate_html(self, limit: int = 5):
        def df_to_html_table(df, limit=5, title=None):
            if df is None:
                return ""

            cols = df.columns
            rows = df.limit(limit).collect()
            parts = []
            if title:
                parts.append(f"<div style='margin:6px 0;font-weight:600'>{html_escape.escape(title)}</div>")
            parts.append("<table class='bordered' style='width:100%; font-size:12px'>")
            parts.append("<thead><tr>" + "".join(f"<th>{html_escape.escape(c)}</th>" for c in cols) + "</tr></thead>")
            parts.append("<tbody>")
            for r in rows:
                parts.append("<tr>" + "".join(f"<td>{html_escape.escape('' if r[c] is None else str(r[c]))}</td>" for c in cols) + "</tr>")
            parts.append("</tbody></table>")
            return "\n".join(parts)

        def icon_for(status: str) -> str:
            return {
                "pass": "check_circle",
                "fail": "cancel",
                "skip": "redo",
                "warning": "warning",
                "info": "info_outline",
            }.get(status, "info_outline")

        def make_step_row(status: str, duration: str, details_html: str) -> str:
            return (
                "<tr class='log' status='{s}'>"
                "<td class='status {s}' title='{s}' alt='{s}'><i class='material-icons'>{ic}</i></td>"
                "<td class='timestamp'>{d}</td>"
                "<td class='step-details'>{details}</td>"
                "</tr>"
            ).format(s=status, ic=icon_for(status), d=duration, details=details_html)

        def pct(passed: int, failed: int) -> str:
            total = passed + failed
            return "0%" if total == 0 else f"{(passed/total)*100:.1f}%"

        def build_status_group(parent_pass: int, parent_fail: int, child_pass: int, child_fail: int, child_skip: int = 0) -> str:
            # Extent v3 usa este objeto para charts
            return (
                "var statusGroup = {\n"
                f"\tpassParent: {parent_pass},\n"
                f"\tfailParent: {parent_fail},\n"
                "\tfatalParent: 0,\n\terrorParent: 0,\n\twarningParent: 0,\n\tskipParent: 0,\n\texceptionsParent: 0,\n"
                f"\tpassChild: {child_pass},\n"
                f"\tfailChild: {child_fail},\n"
                "\tfatalChild: 0,\n\terrorChild: 0,\n\twarningChild: 0,\n"
                f"\tskipChild: {child_skip},\n"
                "\tinfoChild: 0,\n\texceptionsChild: 0,\n"
                "\tpassGrandChild: 0,\n\tfailGrandChild: 0,\n\tfatalGrandChild: 0,\n\terrorGrandChild: 0,\n\twarningGrandChild: 0,\n\tskipGrandChild: 0,\n\tinfoGrandChild: 0,\n\texceptionsGrandChild: 0,\n"
                "};"
            )

        def node_li(node_id: int, node_name: str, status: str, categories: list, desc: str, steps_html: str, start_ts: str, end_ts: str) -> str:
            cats = "".join([f"<span class='category label white-text'>{c}</span>" for c in categories])
            return f"""
                <li class='node level-1 leaf {status}' status='{status}' test-id='{node_id}'>
                <div class='collapsible-header'>
                    <h5 class='node-name'>{node_name}</h5>
                    <span class="node-time label grey lighten-1 white-text">{start_ts}</span>
                    <span class="node-time label grey lighten-1 white-text">{end_ts}</span>
                    <span class='node-duration label grey lighten-1 white-text'>0:00:01.000</span>
                    <span class='test-status right {status}'>{status}</span>
                    {cats}
                    <div class='node-desc'>{desc}</div>
                </div>
                <div class='collapsible-body'>
                    <div class='node-steps'>
                    <table class='bordered table-results'>
                        <thead><tr><th>Status</th><th>Duration</th><th>Details</th></tr></thead>
                        <tbody>
                        {steps_html}
                        </tbody>
                    </table>
                    </div>
                </div>
                </li>
            """

        def build_test_li(table_name: str, start_ts: str, end_ts: str, table_status: str, nodes_html: str) -> str:
            return f"""
                <li class='test displayed active has-leaf {table_status}' status='{table_status}' bdd='false' test-id='1'>
                <div class='test-heading'>
                    <span class='test-name'>{table_name}</span>
                    <span class='test-time'>{start_ts}</span>
                    <span class='test-status right {table_status}'>{table_status}</span>
                </div>
                <div class='test-content hide'>
                    <div class='test-time-info'>
                    <span class='label start-time'>{start_ts}</span>
                    <span class='label end-time'>{end_ts}</span>
                    <span class='label time-taken grey lighten-1 white-text'>0:00:03.000</span>
                    </div>
                    <div class='test-desc'>{table_name}</div>
                    <ul class='collapsible node-list' data-collapsible='accordion'>
                    {nodes_html}
                    </ul>
                </div>
                </li>
            """

        self._settings.sprk = SparkSession.builder.getOrCreate()
        settings = self._settings

        root = f"{settings.output_root}/{settings.applicationid}"
        report_name = self.__tableprops.get("TABLENAME", "TABLE_NAME").replace(".", "_").replace(" ", "_")
        output = f"{root}/html_report/{report_name}.html"
        html_template_path = "abfss://gf-gaa-fmna-nonres-raw@cacgffrefitadlst.dfs.core.windows.net/QA_ATOM_Test/html_template.html"
        
        template_html = "\n".join(row["value"] for row in spark.read.text(html_template_path).toLocalIterator())

        fsr = self.getsummary()
        where_cond = f"where runid='{settings.applicationid}'"
        def load_views_and_get_dfs(qa_folder: str, summary_sql: str, sample_sql: str):
            reader = settings.sprk.read.format(settings.writeformat)
            df = reader.load(f"{settings.errorlog_base}/{qa_folder}")
            settings.createTempViews(df, qa_folder)
            return settings.sprk.sql(summary_sql), settings.sprk.sql(sample_sql)

        results = []
        # 1) Metadata (Column test)
        if self.__tableprops.get("COLTEST", False) and int(fsr.get("COLTEST_ERRORS", 0) or 0) > 0:
            summary_sql = self.__get_sql_report("qa_metadatacomparison", where_cond, "tablename")
            sample_sql  = self.__get_sql_report("qa_metadatacomparison", where_cond)
            s_df, samp_df = load_views_and_get_dfs("qa_metadatacomparison", summary_sql, sample_sql)
            results.append((
                "Column Comparison Test 'On Query'", "fail", ["Column1 Test"],
                "verify table metadata (column count, name, type & position)",
                s_df, samp_df, int(fsr.get("COLTEST_ERRORS", 0) or 0)
            ))
        else:
            # Sin errores o deshabilitado: pass/skip
            status = "skip" if not self.__tableprops.get("COLTEST", False) else "pass"
            results.append((
                "Column Comparison Test 'On Query'", status, ["Column1 Test"],
                "verify table metadata (column count, name, type & position)", None, None, 0
            ))

        # 2) Key comparison
        if self.__tableprops.get("DATATEST", False) and int(fsr.get("KEYTEST_ERRORS", 0) or 0) > 0:
            summary_sql = self.__get_sql_report("qa_keycomparison", where_cond, "tablename, error_msg")
            sample_sql  = self.__get_sql_report("qa_keycomparison", where_cond)
            s_df, samp_df = load_views_and_get_dfs("qa_keycomparison", summary_sql, sample_sql)
            results.append(("Key Comparison Test", "fail", ["Key1 Test"],
                            "verify primary/composite key matching", s_df, samp_df,
                            int(fsr.get("KEYTEST_ERRORS", 0) or 0)))
        else:
            status = "skip" if not self.__tableprops.get("DATATEST", False) else "pass"
            results.append(("Key Comparison Test", status, ["Key1 Test"],
                            "verify primary/composite key matching", None, None, 0))

        # 3) Data comparison
        if self.__tableprops.get("DATATEST", False) and int(fsr.get("DATATEST_ERRORS", 0) or 0) > 0:
            summary_sql = self.__get_sql_report("qa_datacomparison", where_cond, "tablename, failedcol")
            sample_sql  = self.__get_sql_report("qa_datacomparison", where_cond)
            s_df, samp_df = load_views_and_get_dfs("qa_datacomparison", summary_sql, sample_sql)
            results.append(("Data Comparison Test", "fail", ["Data1 Test"],
                            "verify full table row-row & col-col", s_df, samp_df,
                            int(fsr.get("DATATEST_ERRORS", 0) or 0)))
        else:
            status = "skip" if not self.__tableprops.get("DATATEST", False) else "pass"
            results.append(("Data Comparison Test", status, ["Data1 Test"],
                            "verify full table row-row & col-col", None, None, 0))

        now = datetime.now()
        start_ts = now.strftime("%b %d, %Y %H:%M:%S.%f")[:-3]
        end_ts   = start_ts

        child_pass = sum(1 for _, st, *_ in results if st == "pass")
        child_fail = sum(1 for _, st, *_ in results if st == "fail")
        child_skip = sum(1 for _, st, *_ in results if st == "skip")

        table_status = "fail" if child_fail > 0 else "pass"
        parent_pass = 1 if table_status == "pass" else 0
        parent_fail = 1 if table_status == "fail" else 0

        # Nodos (leaf)
        nodes_html = []
        node_id = 2
        for (title, status, cats, desc, summary_df, sample_df, err_count) in results:
            # Details: label + tablas si existen
            label = f"<span class='label white-text {'red' if status == 'fail' else 'green' if status == 'pass' else 'orange'}'>" \
                    f"<b>Errors:</b> <b><u>{err_count}</u></b></span>"

            tables = ""
            if status == "fail" and summary_df is not None and sample_df is not None:
                tables = (
                    "<div style='margin-top:8px'>"
                    + df_to_html_table(summary_df, limit=limit, title=f"{title} Summary")
                    + "<div style='height:10px'></div>"
                    + df_to_html_table(sample_df, limit=limit, title=f"{title} Samples")
                    + "</div>"
                )
            elif status == "skip":
                tables = "<div class='label white-text orange'><b>Test not executed.</b></div>"

            step = make_step_row(status, "0:00:01.000", label + tables)
            nodes_html.append(node_li(node_id, title, status, cats, desc, step, start_ts, end_ts))
            node_id += 1

        nodes_html = "\n".join(nodes_html)

        # Parent test
        test_name = self.__tableprops.get("TABLENAME", "TABLE_NAME").replace(".", "_").replace(" ", "_")
        test_li = build_test_li(test_name, start_ts, end_ts, table_status, nodes_html)

        status_group_js = build_status_group(parent_pass, parent_fail, child_pass, child_fail, child_skip)

        # -------------------------
        # Dashboard tables (Environment/Categories)
        # -------------------------
        dash_env_rows = "\n".join([
            f"<tr><td>RunID</td><td>{html_escape.escape(settings.applicationid)}</td></tr>",
            f"<tr><td>Env</td><td>Python & Azure Databricks</td></tr>",
            f"<tr><td>Python</td><td>{html_escape.escape(settings.CODE_VER if hasattr(settings, 'CODE_VER') else '')}</td></tr>",
        ])

        # Categories: for now 1 row per type of test 
        col_pass = 1 if results[0][1] == "pass" else 0
        col_fail = 1 if results[0][1] == "fail" else 0

        key_pass = 1 if results[1][1] == "pass" else 0
        key_fail = 1 if results[1][1] == "fail" else 0

        data_pass = 1 if results[2][1] == "pass" else 0
        data_fail = 1 if results[2][1] == "fail" else 0

        dash_category_rows = "\n".join([
            f"<tr><td>Column Comparison Test</td><td>{col_pass}</td><td>{col_fail}</td><td>{pct(col_pass, col_fail)}</td></tr>",
            f"<tr><td>Key Comparison Test</td><td>{key_pass}</td><td>{key_fail}</td><td>{pct(key_pass, key_fail)}</td></tr>",
            f"<tr><td>Data Comparison Test</td><td>{data_pass}</td><td>{data_fail}</td><td>{pct(data_pass, data_fail)}</td></tr>",
            f"<tr><td>Dataquality Test</td><td>0</td><td>0</td><td>0%</td></tr>",
        ])

        # placeholders replacements + write
        out = template_html
        out = out.replace("{{REPORT_NAME}}", "ATOM Test Report - Generated")
        out = out.replace("{{REPORT_DATE_LABEL}}", now.strftime("%B %d, %Y"))
        out = out.replace("{{REPORT_VERSION_LABEL}}", "3.1.5")

        out = out.replace("{{STATUS_GROUP_JS}}", status_group_js)
        out = out.replace("{{TEST_COLLECTION_LI}}", test_li)

        out = out.replace("{{PARENT_TOTAL}}", str(parent_pass + parent_fail))
        out = out.replace("{{CHILD_TOTAL}}", str(child_pass + child_fail + child_skip))

        out = out.replace("{{PARENT_PASS}}", str(parent_pass))
        out = out.replace("{{PARENT_FAIL}}", str(parent_fail))
        out = out.replace("{{CHILD_PASS}}", str(child_pass))
        out = out.replace("{{CHILD_FAIL}}", str(child_fail))

        out = out.replace("{{START_TIME}}", start_ts)
        out = out.replace("{{END_TIME}}", end_ts)
        out = out.replace("{{TIME_TAKEN}}", "0:00:03.000")

        out = out.replace("{{DASH_ENV_ROWS}}", dash_env_rows)
        out = out.replace("{{DASH_CATEGORY_ROWS}}", dash_category_rows)

        # Create and write a ABFSS/DBFS
        dbutils.fs.mkdirs(output.rsplit("/", 1)[0])
        dbutils.fs.put(output, out, True)

        print(f"✅ HTML report generated: {output}")
        #self._settings.cleartempviews()
        return output

    # log
    def logTestRuns(self):
        Writer.clear()
        # tablename
        Writer.add(self.__resdetailsummary['TABLE_NAME'])
        # starttime
        Writer.add(self.__resdetailsummary['START_TIME']["log"])
        # endtime
        Writer.add(self.__resdetailsummary['END_TIME']["log"])

        # metadata_test
        col_test = self.__resdetailsummary["COL_TEST"]
        data_test = self.__resdetailsummary["DATA_TEST"]
        key_test = self.__resdetailsummary["KEY_TEST"]

        Writer.add(Atom_UDF.iif(col_test['START_STATUS'] and col_test['END_STATUS'], 'Y', 'N'))
        # datacomparision_test
        Writer.add(Atom_UDF.iif((key_test['START_STATUS'] or data_test['START_STATUS']) and
                           (key_test['END_STATUS'] or data_test['END_STATUS']), 'Y', 'N'))
        # metadatatested_src
        Writer.add(Atom_UDF.iif(col_test['START_STATUS'] and col_test['END_STATUS'], str(col_test['SRC_COL_COUNT']), '-1'))

        # metadatatested_trg
        Writer.add(Atom_UDF.iif(col_test['START_STATUS'] and col_test['END_STATUS'], str(col_test['TRG_COL_COUNT']), '-1'))
        # metadata_errors
        Writer.add(Atom_UDF.iif(col_test['START_STATUS'] and col_test['END_STATUS'], str(col_test['COLTEST_ERRORS']), '-1'))
        # colscount_src
        Writer.add(Atom_UDF.iif(data_test['START_STATUS'] and data_test['END_STATUS'], str(data_test['SRC_COL_COUNT']), '-1'))
        # recordcount_src
        Writer.add(Atom_UDF.iif(data_test['START_STATUS'] and data_test['END_STATUS'], str(data_test['SRC_REC_COUNT']), '-1'))
        # colscount_trg
        Writer.add(Atom_UDF.iif(data_test['START_STATUS'] and data_test['END_STATUS'], str(data_test['TRG_COL_COUNT']), '-1'))
        # recordcount_trg
        Writer.add(Atom_UDF.iif(data_test['START_STATUS'] and data_test['END_STATUS'], str(data_test['TRG_REC_COUNT']), '-1'))
        # key_errors
        Writer.add(Atom_UDF.iif(key_test['START_STATUS'] and key_test['END_STATUS'], str(key_test['KEYTEST_ERRORS']), '-1'))

        #failed cols
        Writer.add(Atom_UDF.iif(data_test['START_STATUS'] and data_test['END_STATUS'], str(data_test['FAILED_COLS']), '-1'))

        # data_errorrs
        Writer.add(Atom_UDF.iif(data_test['START_STATUS'] and data_test['END_STATUS'], str(data_test['DATATEST_ERRORS']), '-1'))

        # no of data quality test done
        Writer.add(Atom_UDF.iif(self.__ressummary['DATAQUALITY_TESTS'] == -1, '-1', str(self.__ressummary['DATAQUALITY_TESTS'])))

        # no of data quality test passed
        Writer.add(Atom_UDF.iif(self.__ressummary['DATAQUALITY_TESTS'] == -1, '-1', str(self.__ressummary['DATAQUALITY_TEST_PASSED'])))

        # no of data quality test failed
        Writer.add(Atom_UDF.iif(self.__ressummary['DATAQUALITY_TESTS'] == -1, '-1', str(self.__ressummary['DATAQUALITY_TEST_FAILED'])))

        # data quality errors
        Writer.add(Atom_UDF.iif(self.__ressummary['DATAQUALITY_ERRORS'] == -1, '-1', str(self.__ressummary['DATAQUALITY_ERRORS'])))
        # runid
        Writer.add(self._settings.applicationid)
        Writer.qa_testruns()
        return


"""
#issues
when date is given as pk need to rep the issue
nimpha issues need to rep the issue
variable in python script---- done--- tobe tested
"""

"""
Geethika: column count mismatch showing as 2 but there are more cols
Jagrati: no pk first col is null and it does not concat but concat_ws works
Veera: count mis matches 
"""

#writer.py
class Writer:
    # writeFormat = "delta"
    _settings:Optional[SettingsProto] = None
    linenum = 0
    fields = []
    logger_schema = T.StructType([
        T.StructField("id", T.StringType()),
        T.StructField("text", T.StringType())
    ])

    qa_testinfo_schema = T.StructType([
        T.StructField("runid", T.StringType()),
        T.StructField("starttime", T.StringType()),
        T.StructField("project", T.StringType()),
        T.StructField("project_desc", T.StringType()),
        T.StructField("username", T.StringType()),
        T.StructField("ver", T.StringType()),
        T.StructField("runyear", T.StringType())
    ])

    qa_testruns_schema = T.StructType([
        T.StructField("tablename", T.StringType()),
        T.StructField("starttime", T.StringType()),
        T.StructField("endtime", T.StringType()),
        T.StructField("metadata_test", T.StringType()),
        T.StructField("datacomparision_test", T.StringType()),
        T.StructField("metadatatested_src", T.StringType()),
        T.StructField("metadatatested_trg", T.StringType()),
        T.StructField("metadata_errors", T.StringType()),
        T.StructField("colscount_src", T.StringType()),
        T.StructField("recordcount_src", T.StringType()),
        T.StructField("colscount_trg", T.StringType()),
        T.StructField("recordcount_trg", T.StringType()),
        T.StructField("key_errors", T.StringType()),
        T.StructField("failedcols_datatest", T.StringType()),
        T.StructField("data_errors", T.StringType()),
        T.StructField("dataquality_tests", T.StringType()),
        T.StructField("dataquality_test_passed", T.StringType()),
        T.StructField("dataquality_test_failed", T.StringType()),
        T.StructField("dataquality_errors", T.StringType()),
        T.StructField("runid", T.StringType())
    ])

    qa_testinfosummary_schema = T.StructType([
        T.StructField("runid", T.StringType()),
        T.StructField("starttime", T.StringType()),
        T.StructField("endtime", T.StringType())
    ])

    @classmethod
    def set_settings(cls, settings:SettingsProto) -> None:
        """Set the Settings used by every Writer method."""
        cls._settings = settings
        return

    @staticmethod
    def add(o):
        """Append a value to the row built for the next summary write."""
        Writer.fields.append(o)
        return

    @staticmethod
    def clear():
        """Clear the row being built."""
        Writer.fields = []

    @classmethod
    def logger(cls, log, printtoconsole=False):
        """Print *log* when *printtoconsole* is True and, with loglevel "all" or "summary", append it as
        one row to {GFDOQA_TEST_PATH}/{applicationid}/qalog. Errors are printed, not raised.

        The text column is a string: pass text (use str(e) for exceptions).
        """
        try:
            logLevel = cls._settings.loglevel
            if printtoconsole:
                print(log)

            if logLevel.lower() in ['all', 'summary']:
                records = [[str(Writer.linenum), log]]
                cls._settings.sprk.createDataFrame(records, Writer.logger_schema) \
                    .withColumn("id", F.col("id").cast(T.LongType())) \
                    .write.format(cls._settings.writeformat).mode("append") \
                    .save(f"{GFDOQA_TEST_PATH}/{cls._settings.applicationid}/qalog")
                Writer.linenum += 1

        except Exception as e:
            print(e)
        return

    @classmethod
    def qa_testinfo(cls):
        """Write the row built with add() to {summarylog_base}/qa_testinfo (partitioned by runyear). Skipped with loglevel "none"."""
        try:
            logLevel = cls._settings.loglevel
            if logLevel.lower() in ['all', 'summary']:
                print("writing to qa_testinfo...")
                records = [Writer.fields]
                Writer.fields = []

                cls._settings.sprk.createDataFrame(records, Writer.qa_testinfo_schema) \
                    .withColumn("starttime", F.col("starttime").cast(T.TimestampType())) \
                    .withColumn("runyear", F.col("runyear").cast(T.LongType())) \
                    .write.partitionBy("runyear") \
                    .format(cls._settings.writeformat) \
                    .mode("append") \
                    .save(f"{cls._settings.summarylog_base}/qa_testinfo")
            else:
                print("writing to qa_testinfo skipped...")
        except Exception as e:
            Writer.logger("exception in writer.qa_testinfo")
            Writer.logger(e)
        return

    @classmethod
    def qa_testruns(cls):
        """Write the row built with add() to {summarylog_base}/qa_testruns (partitioned by runid). Skipped with loglevel "none"."""
        try:
            logLevel = cls._settings.loglevel
            if logLevel.lower() in ['all', 'summary']:
                print("writing to qa_testruns...")
                records = [Writer.fields]
                Writer.fields = []

                casts = {
                    "starttime": T.TimestampType(),
                    "endtime": T.TimestampType(),
                    "metadatatested_src": T.LongType(),
                    "metadatatested_trg": T.LongType(),
                    "metadata_errors": T.LongType(),
                    "colscount_src": T.LongType(),
                    "recordcount_src": T.LongType(),
                    "colscount_trg": T.LongType(),
                    "recordcount_trg": T.LongType(),
                    "key_errors": T.LongType(),
                    "failedcols_datatest": T.LongType(),
                    "data_errors": T.LongType(),
                    "dataquality_tests": T.LongType(),
                    "dataquality_test_passed": T.LongType(),
                    "dataquality_test_failed": T.LongType(),
                    "dataquality_errors": T.LongType(),
                }

                df = cls._settings.sprk.createDataFrame(records, Writer.qa_testruns_schema)
                for col_name, data_type in casts.items():
                    df = df.withColumn(col_name, F.col(col_name).cast(data_type))

                df.write.partitionBy("runid").format(cls._settings.writeformat) \
                    .mode("append").save(f"{cls._settings.summarylog_base}/qa_testruns")
            else:
                print("writing to qa_testruns skipped...")
        except Exception as e:
            Writer.logger(f"Exception in writer.qa_testruns\n{e}")
        return

    @classmethod
    def qa_testinfosummary(cls, summarInfo):
        """Write runid, start and end time of the run to {summarylog_base}/qa_testinfosummary. Skipped with loglevel "none"."""
        try:
            logLevel = cls._settings.loglevel
            if logLevel.lower() in ['all', 'summary']:
                print("writing to qa_testinfosummary...")
                Writer.clear()
                Writer.add(cls._settings.applicationid)
                #Writer.add(summarInfo['FILE_NAME'])
                Writer.add(summarInfo['START_TIME']["log"])
                Writer.add(summarInfo['END_TIME']["log"])
                #Writer.add(summarInfo['TABLE_COUNT'])
                #Writer.add(summarInfo['MD_SRC_COL_COUNT'])
                #Writer.add(summarInfo['MD_TRG_COL_COUNT'])
                #Writer.add(summarInfo['COLTEST_ERRORS'])
                #Writer.add(summarInfo['DT_SRC_COL_COUNT'])
                #Writer.add(summarInfo['SRC_REC_COUNT'])
                #Writer.add(summarInfo['DT_TRG_COL_COUNT'])
                #Writer.add(summarInfo['TRG_REC_COUNT'])
                #Writer.add(summarInfo['KEYTEST_ERRORS'])
                #Writer.add(summarInfo['DATATEST_ERRORS'])
                #Writer.add(summarInfo['DATAQUALITY_ERRORS'])
                #Writer.add(summarInfo['START_TIME']["log"][:4])

                records = [Writer.fields]
                Writer.fields = []
                cls._settings.sprk.createDataFrame(records, Writer.qa_testinfosummary_schema) \
                    .withColumn("starttime", F.col("starttime").cast(T.TimestampType())) \
                    .withColumn("endtime", F.col("endtime").cast(T.TimestampType())) \
                    .write.partitionBy("runid") \
                    .format(cls._settings.writeformat) \
                    .mode("append") \
                    .save(f"{cls._settings.summarylog_base}/qa_testinfosummary")
            else:
                print("writing to qa_testinfosummary skipped...")
        except Exception as e:
            Writer.logger(f"Exception in writer.qa_testinfosummary\n{e}")

    @classmethod
    def qa_metadatacomparison(cls, dataTable:DataFrame, tp:dict):
        try:
            logLevel = cls._settings.loglevel
            if logLevel.lower() == 'all':
                print("writing to qa_metadatacomparison...")
                tableName = tp["TABLENAME"]  # Table.name
                appID = cls._settings.applicationid
                maxError = cls._settings.maxerror
                cls._settings.createTempViews(dataTable, f"col_errors_{tableName}")

                dataTable.withColumn("tablename", F.lit(tableName)).withColumn("runid", F.lit(appID)) \
                    .write.format(cls._settings.writeformat) \
                    .partitionBy('runid') \
                    .mode('append') \
                    .save(f"{cls._settings.errorlog_base}/qa_metadatacomparison")
            else:
                print("writing to qa_metadatacomparison skipped...")

            cls._settings.cleartempview(f"col_errors_{tableName}")
            del dataTable
            gc.collect()
        except Exception as e:
            del dataTable
            gc.collect()
            Writer.logger("exception in writer.qa_metadatacomparison")
            Writer.logger(e)

    @classmethod
    def qa_keycomparison(cls, dataTable, tp):
        try:
            logLevel = cls._settings.loglevel
            if logLevel.lower() == 'all':
                print("writing to qa_keycomparison...")
                tableName = tp["TABLENAME"]  # Table.name
                appID = cls._settings.applicationid
                maxError = cls._settings.maxerror

                cls._settings.setsparkobject(SparkSession.builder.getOrCreate())
                dataTable.createOrReplaceTempView(f"key_errors_{tableName}")
                #cls._settings.createTempViews(dataTable, f"key_errors_{tableName}")
                if int(maxError) < 0:
                    print("Creating tmp df for results in qa_keycomparison")
                    dataTable.withColumnRenamed("keycol", "pkey").withColumnRenamed("errortype", "error_msg") \
                        .withColumn("runid", F.lit(appID)).withColumn("tablename", F.lit(tableName)) \
                        .write.partitionBy('runid').format(cls._settings.writeformat) \
                        .mode('append').save(f"{cls._settings.errorlog_base}/qa_keycomparison")
                else:
                    spark.sql("with k as (select row_number() over(partition by  a.errortype order by "
                             f"a.errortype) as rn, '{tableName}' as tablename, "
                             "cleantabledelimiters(a.keycol) as pkey, "
                             f"cleantabledelimiters(a.errortype) as error_msg, '{appID}' as runid "
                             f"from key_errors_{tableName} a) select tablename, pkey, error_msg, runid from k "
                             f"where rn<={str(maxError).replace(',', '')}").write.partitionBy('runid') \
                        .format(cls._settings.writeformat).mode('append') \
                        .save(f"{cls._settings.errorlog_base}/qa_keycomparison")
            else:
                print("writing to qa_keycomparison skipped...")

            cls._settings.cleartempview(f"key_errors_{tableName}")
            del dataTable
            gc.collect()
        except Exception as e:
            del dataTable
            gc.collect()
            Writer.logger("exception in writer.qa_keycomparison")
            Writer.logger(e)
            print(e)
        return

    @classmethod
    def qa_datacomparison(cls, dataTable:DataFrame, tp:dict):
        try:
            logLevel = cls._settings.loglevel
            if logLevel.lower() == 'all':
                print("writing to qa_datacomparison...")
                tableName = tp["TABLENAME"]  # Table.name
                appID = cls._settings.applicationid
                n_samples = int(cls._settings.maxerror)

                # dataTable.createOrReplaceTempView(f"data_errors_{tableName}")
                cls._settings.createTempViews(dataTable, f"data_errors_{tableName}")

                if n_samples:
                    dataTable.withColumnRenamed("field_name", "failedcol") \
                        .withColumnRenamed("keycol", "pkey") \
                        .withColumnRenamed("source_data", "src_data") \
                        .withColumnRenamed("target_data", "trg_data") \
                        .withColumn("runid", F.lit(appID)) \
                        .withColumn("tablename", F.lit(tableName)) \
                        .write.format(cls._settings.writeformat) \
                        .partitionBy('runid').mode('append') \
                        .save(f"{cls._settings.errorlog_base}/qa_datacomparison")
                else:
                    max_val = str(n_samples).replace(',', '')
                    clen_fn = lambda s: f"cleantabledelimiters({s})"
                    query = f"""
                    SELECT tablename, failedcol, pkey, src_data, trg_data, runid
                    FROM (
                        SELECT
                            ROW_NUMBER() OVER (PARTITION BY a.field_name ORDER BY a.field_name) AS rn,
                            '{tableName}' AS tablename,
                            {clean_fn("a.field_name")} AS failedcol,
                            {clean_fn("a.keycol")} AS pkey,
                            {clean_fn("a.source_data")} AS src_data,
                            {clean_fn("a.target_data")} AS trg_data,
                            '{appID}' AS runid
                        FROM data_errors_{tableName} a
                    ) d
                    WHERE rn <= {max_val}
                    """

                    cls._settings.sprk.sql(query) \
                        .write.format(cls._settings.writeformat) \
                        .partitionBy('runid').mode('append') \
                        .save(f"{cls._settings.errorlog_base}/qa_datacomparison")
            else:
                print("writing to qa_datacomparison skipped...")

            cls._settings.cleartempview(f"data_errors_{tableName}")
        
        except Exception as e:
            Writer.logger(f"Exception in writer.qa_datacomparison: {e}")
        
        del dataTable
        gc.collect()
        return

    @classmethod
    def qa_dataquality(cls, dataTable:DataFrame, dqname:str):
        """Append the failed rows of a data quality check to {errorlog_base}/qa_dataquality/{dqname},
        limited to maxerror rows (negative = all). Only with loglevel "all".
        """
        try:
            logLevel = cls._settings.loglevel
            if logLevel.lower() == 'all':
                print("writing to qa_dataquality...")
                n_sample = int(cls._settings.maxerror)
                tmp_df = dataTable if n_sample<0 else dataTable.limit(n_sample)

                path = f"{cls._settings.errorlog_base}/qa_dataquality/{dqname}"
                tmp_df.write.format(cls._settings.writeformat).mode('append').save(path)
            else:
                print("writing to qa_dataquality skipped...")

        except Exception as e:
            Writer.logger(f"Exception in writer.qa_dataquality {e}")
        
        del dataTable
        gc.collect()
        return
#writer.py